In [ ]:
import os
base_dir = "/kaggle/working/"
os.chdir(base_dir)
os.makedirs("TextSummarization/data/raw", exist_ok=True)
os.makedirs("TextSummarization/data/processed", exist_ok=True)
os.makedirs("TextSummarization/preprocessing", exist_ok=True)
os.makedirs("TextSummarization/results", exist_ok=True)
import shutil

# Define source and destination paths
input_dir = "/kaggle/input/text-summarization1/"
raw_dir = "TextSummarization/data/raw/"

# List of files to transfer
files = ["train.csv", "test.csv", "validation.csv"]

# Copy each file
for file in files:
    src = os.path.join(input_dir, file)
    dst = os.path.join(raw_dir, file)
    shutil.copy(src, dst)

print("Files successfully copied to raw directory.")

In [2]:
!pip install evaluate contractions textstat rouge_score bert_score transformers seqeval --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 

In [ ]:
import re
import pandas as pd
from tqdm import tqdm
from contractions import fix

def clean_text(text: str) -> str:
    """
    Cleans a single string by:
    1. Lowercasing the text.
    2. Expanding contractions (e.g., "don't" -> "do not").
    3. Removing HTML tags.
    4. Removing special characters and extra spaces.

    Args:
        text (str): The input text to clean.

    Returns:
        str: The cleaned text.
    """
    if not isinstance(text, str):
        return "" # Return empty string for non-string inputs, or handle as needed
    text = text.lower()
    text = fix(text)  # Expand contractions
    text = re.sub(r'<[^>]+>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

def normalize_data(input_path: str, output_path: str, sample_size: int = None):
    """
    Normalizes the 'article' and 'highlights' columns in a CSV file.

    Args:
        input_path (str): Path to the raw CSV file.
        output_path (str): Path to save the normalized CSV file.
        sample_size (int, optional): Number of rows to sample. If None, process all rows.
    """
    print(f"Normalizing data from {input_path}...")
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}")
        return
    except Exception as e:
        print(f"Error reading CSV {input_path}: {e}")
        return

    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"  Sampling {sample_size} rows.")

    # Apply cleaning to 'article' and 'highlights' columns
    tqdm.pandas(desc="Cleaning articles")
    df['article'] = df['article'].progress_apply(clean_text)

    tqdm.pandas(desc="Cleaning highlights")
    df['highlights'] = df['highlights'].progress_apply(clean_text)

    try:
        df.to_csv(output_path, index=False)
        print(f"Normalized data saved to {output_path}")
    except Exception as e:
        print(f"Error saving normalized data to {output_path}: {e}")

if __name__ == "__main__":
    # Define paths (relative to /kaggle/working/TextSummarization/)
    BASE_DIR = "/kaggle/working/TextSummarization"
    RAW_DATA_DIR = f"{BASE_DIR}/data/raw"
    PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"

    # Ensure directories exist
    import os
    os.makedirs(RAW_DATA_DIR, exist_ok=True)
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    # Normalize train, test, and validation datasets
    # For training and validation, use a sample size as specified in the prompt
    normalize_data(f"{RAW_DATA_DIR}/train.csv", f"{PROCESSED_DATA_DIR}/train_normalized.csv", sample_size=20000)
    normalize_data(f"{RAW_DATA_DIR}/validation.csv", f"{PROCESSED_DATA_DIR}/validation_normalized.csv", sample_size=20000)
    # For test, use the full dataset
    normalize_data(f"{RAW_DATA_DIR}/test.csv", f"{PROCESSED_DATA_DIR}/test_normalized.csv", sample_size=None)


In [ ]:
import os
import pandas as pd
import sentencepiece as spm
import torch
from tqdm import tqdm

# Define constants
MAX_LEN = 64
VOCAB_SIZE = 30000

# Define paths
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model"
TOKENIZER_VOCAB_PATH = f"{BASE_DIR}/tokenizer.vocab"

def train_sentencepiece_tokenizer(data_paths: list[str], model_prefix: str, vocab_size: int):
    """
    Trains a SentencePiece tokenizer on the combined text from specified data paths.

    Args:
        data_paths (list[str]): List of paths to CSV files containing 'article' and 'highlights'.
        model_prefix (str): Prefix for the SentencePiece model files.
        vocab_size (int): Desired vocabulary size.
    """
    print(f"Training SentencePiece tokenizer with vocab size {vocab_size}...")
    temp_text_file = f"{model_prefix}_temp_corpus.txt"

    # Combine all text from articles and highlights into a single file
    all_text = []
    for path in data_paths:
        try:
            df = pd.read_csv(path)
            all_text.extend(df['article'].dropna().tolist())
            all_text.extend(df['highlights'].dropna().tolist())
        except FileNotFoundError:
            print(f"Warning: File not found at {path}. Skipping.")
        except Exception as e:
            print(f"Error reading CSV {path}: {e}")
            continue

    if not all_text:
        print("Error: No text data found to train tokenizer.")
        return

    with open(temp_text_file, 'w', encoding='utf-8') as f:
        for text in tqdm(all_text, desc="Writing corpus for tokenizer training"):
            f.write(text + '\n')

    # Train SentencePiece model
    spm.SentencePieceTrainer.train(
        f'--input={temp_text_file} --model_prefix={model_prefix} '
        f'--vocab_size={vocab_size} --character_coverage=1.0 '
        f'--model_type=unigram --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3'
    )
    print(f"SentencePiece tokenizer trained and saved as {model_prefix}.model")
    os.remove(temp_text_file) # Clean up temporary file

def load_tokenizer(model_path: str):
    """
    Loads a pre-trained SentencePiece tokenizer.

    Args:
        model_path (str): Path to the SentencePiece model file.

    Returns:
        spm.SentencePieceProcessor: Loaded tokenizer.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Tokenizer model not found at {model_path}")
    sp = spm.SentencePieceProcessor()
    sp.load(model_path)
    return sp

def tokenize_data(sp_tokenizer, input_path: str, output_path: str, max_len: int):
    """
    Tokenizes 'article' and 'highlights' columns using the provided tokenizer,
    pads/truncates sequences, and saves the data as a PyTorch tensor.

    Args:
        sp_tokenizer: Trained SentencePiece tokenizer.
        input_path (str): Path to the normalized CSV file.
        output_path (str): Path to save the tokenized data as a PyTorch .pt file.
        max_len (int): Maximum sequence length for padding/truncation.
    """
    print(f"Tokenizing data from {input_path}...")
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}. Skipping tokenization.")
        return
    except Exception as e:
        print(f"Error reading CSV {input_path}: {e}")
        return

    articles_tokenized = []
    highlights_tokenized = []

    # Tokenize and pad/truncate
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing and padding"):
        article_ids = sp_tokenizer.encode_as_ids(str(row['article']))
        highlights_ids = sp_tokenizer.encode_as_ids(str(row['highlights']))

        # Add EOS token and pad/truncate
        article_ids = article_ids[:max_len-1] + [sp_tokenizer.eos_id()]
        highlights_ids = highlights_ids[:max_len-1] + [sp_tokenizer.eos_id()]

        article_ids = article_ids + [sp_tokenizer.pad_id()] * (max_len - len(article_ids))
        highlights_ids = highlights_ids + [sp_tokenizer.pad_id()] * (max_len - len(highlights_ids))

        articles_tokenized.append(article_ids[:max_len])
        highlights_tokenized.append(highlights_ids[:max_len])

    # Convert to PyTorch tensors
    articles_tensor = torch.tensor(articles_tokenized, dtype=torch.long)
    highlights_tensor = torch.tensor(highlights_tokenized, dtype=torch.long)

    # Save as .pt file
    processed_data = {
        'articles': articles_tensor,
        'highlights': highlights_tensor,
        'pad_id': sp_tokenizer.pad_id(),
        'bos_id': sp_tokenizer.bos_id(),
        'eos_id': sp_tokenizer.eos_id(),
        'vocab_size': sp_tokenizer.get_piece_size()
    }
    try:
        torch.save(processed_data, output_path)
        print(f"Tokenized data saved to {output_path}")
    except Exception as e:
        print(f"Error saving tokenized data to {output_path}: {e}")


if __name__ == "__main__":
    # Ensure directories exist (redundant but good for standalone execution)
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    # Paths to normalized data
    train_normalized_path = f"{PROCESSED_DATA_DIR}/train_normalized.csv"
    validation_normalized_path = f"{PROCESSED_DATA_DIR}/validation_normalized.csv"
    test_normalized_path = f"{PROCESSED_DATA_DIR}/test_normalized.csv"

    # Train tokenizer using combined train and validation normalized data
    # (Using normalized data as corpus for tokenizer training)
    train_sentencepiece_tokenizer(
        data_paths=[train_normalized_path, validation_normalized_path],
        model_prefix=f"{BASE_DIR}/tokenizer",
        vocab_size=VOCAB_SIZE
    )

    # Load the trained tokenizer
    sp_tokenizer_model = load_tokenizer(TOKENIZER_MODEL_PATH)

    # Tokenize and save each dataset
    tokenize_data(sp_tokenizer_model, train_normalized_path, f"{PROCESSED_DATA_DIR}/train_tokenized.pt", MAX_LEN)
    tokenize_data(sp_tokenizer_model, validation_normalized_path, f"{PROCESSED_DATA_DIR}/validation_tokenized.pt", MAX_LEN)
    tokenize_data(sp_tokenizer_model, test_normalized_path, f"{PROCESSED_DATA_DIR}/test_tokenized.pt", MAX_LEN)

    # Example of how to get special token IDs for later use
    print(f"PAD ID: {sp_tokenizer_model.pad_id()}")
    print(f"BOS ID: {sp_tokenizer_model.bos_id()}")
    print(f"EOS ID: {sp_tokenizer_model.eos_id()}")
    print(f"VOCAB SIZE: {sp_tokenizer_model.get_piece_size()}")


In [ ]:
import os
import pandas as pd
import spacy
import torch
import sentencepiece as spm
from tqdm import tqdm

# Define constants
MAX_LEN = 64
VOCAB_SIZE = 30000 # Should match tokenizer training
PAD_ID = 0 # SentencePiece default pad_id
BOS_ID = 2 # SentencePiece default bos_id
EOS_ID = 3 # SentencePiece default eos_id

# Define paths
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model"
SPACY_MODEL = "en_core_web_sm" # Using a small model for faster processing

# Load spaCy model
try:
    nlp = spacy.load(SPACY_MODEL)
    print(f"spaCy model '{SPACY_MODEL}' loaded successfully.")
except OSError:
    print(f"Downloading spaCy model '{SPACY_MODEL}'...")
    os.system(f"python -m spacy download {SPACY_MODEL}")
    nlp = spacy.load(SPACY_MODEL)
    print(f"spaCy model '{SPACY_MODEL}' downloaded and loaded.")

def load_tokenizer(model_path: str):
    """
    Loads a pre-trained SentencePiece tokenizer.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Tokenizer model not found at {model_path}")
    sp = spm.SentencePieceProcessor()
    sp.load(model_path)
    return sp

def create_tag_mappings(nlp_model):
    """
    Creates integer mappings for POS, DEP, and NER tags from spaCy's vocabulary.
    This function now ensures 'O' (Outside) tag for NER is explicitly handled.
    """
    pos_labels = list(nlp_model.pipe_labels.get('tagger', []))
    dep_labels = list(nlp_model.pipe_labels.get('parser', []))
    
    ner_labels = []
    if 'ner' in nlp_model.pipe_names:
        ner_labels = list(nlp_model.get_pipe('ner').labels)

    # Add 'O' (Outside) tag explicitly if not present, and ensure it gets a consistent ID
    # Sort labels to ensure consistent ID assignment across runs
    unique_ner_labels = sorted(list(set(ner_labels + ['O'])))
    ner_map = {tag: i for i, tag in enumerate(unique_ner_labels)}
    
    pos_map = {tag: i for i, tag in enumerate(sorted(list(set(pos_labels))))}
    dep_map = {tag: i for i, tag in enumerate(sorted(list(set(dep_labels))))}

    return pos_map, dep_map, ner_map

def process_and_tokenize_data(sp_tokenizer, input_path: str, output_path: str, max_len: int,
                              pos_map: dict, dep_map: dict, ner_map: dict):
    """
    Processes 'article' and 'highlights' columns:
    1. Applies spaCy to extract POS, DEP, and NER tags (word-level).
    2. Tokenizes text using SentencePiece.
    3. Pads/truncates all sequences (text, POS, DEP, NER) to max_len.
    4. Saves the data as a PyTorch .pt file.
    """
    print(f"Processing and tokenizing data from {input_path}...")
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}. Skipping processing.")
        return
    except Exception as e:
        print(f"Error reading CSV {input_path}: {e}")
        return

    all_articles_ids = []
    all_highlights_ids = []
    all_articles_pos_ids = []
    all_highlights_pos_ids = []
    all_articles_dep_ids = []
    all_highlights_dep_ids = []
    # New lists for word-level NER tags (important for BERT alignment)
    all_articles_ner_tags_word_level = [] 
    all_highlights_ner_tags_word_level = []

    # Get default ID for unknown POS/DEP tags (assuming 0 is PAD_ID)
    UNK_TAG_ID = 0 # If 0 is PAD_ID, use a different ID for UNK if needed, or PAD_ID is okay if handled by mask
    O_NER_ID = ner_map['O'] # Get the ID for 'O' tag

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing with spaCy & Tokenizing"):
        article_text = str(row['article'])
        highlights_text = str(row['highlights'])

        # --- Process Article ---
        doc_article = nlp(article_text)
        
        # SentencePiece token IDs for articles
        article_sp_ids = sp_tokenizer.encode_as_ids(article_text)

        # Word-level POS, DEP, and NER tags from spaCy
        current_article_pos_ids_word_level = []
        current_article_dep_ids_word_level = []
        current_article_ner_ids_word_level = [] # This is the list we'll store

        for token in doc_article:
            current_article_pos_ids_word_level.append(pos_map.get(token.pos_, UNK_TAG_ID))
            current_article_dep_ids_word_level.append(dep_map.get(token.dep_, UNK_TAG_ID))
            
            # Get NER tag for each spaCy token
            ner_tag_found = False
            for ent in doc_article.ents:
                if token.i >= ent.start and token.i < ent.end: # Check if token is within entity span
                    # For multi-token entities, spaCy provides the entity label for each token within the entity.
                    # We are mapping B- and I- tags later in NERDatasetBERT for subword alignment.
                    # Here, we store the full entity label for any token belonging to it.
                    current_article_ner_ids_word_level.append(ner_map.get(ent.label_, O_NER_ID))
                    ner_tag_found = True
                    break
            if not ner_tag_found:
                current_article_ner_ids_word_level.append(O_NER_ID) # 'O' for outside an entity

        # --- Process Highlights (similar logic) ---
        doc_highlights = nlp(highlights_text)
        highlights_sp_ids = sp_tokenizer.encode_as_ids(highlights_text)

        current_highlights_pos_ids_word_level = []
        current_highlights_dep_ids_word_level = []
        current_highlights_ner_ids_word_level = []

        for token in doc_highlights:
            current_highlights_pos_ids_word_level.append(pos_map.get(token.pos_, UNK_TAG_ID))
            current_highlights_dep_ids_word_level.append(dep_map.get(token.dep_, UNK_TAG_ID))

            ner_tag_found = False
            for ent in doc_highlights.ents:
                if token.i >= ent.start and token.i < ent.end:
                    current_highlights_ner_ids_word_level.append(ner_map.get(ent.label_, O_NER_ID))
                    ner_tag_found = True
                    break
            if not ner_tag_found:
                current_highlights_ner_ids_word_level.append(O_NER_ID)

        # Helper to pad/truncate sequences
        def pad_or_truncate(seq, target_len, pad_value):
            if len(seq) > target_len:
                return seq[:target_len]
            return seq + [pad_value] * (target_len - len(seq))


        # Apply padding/truncation for SentencePiece IDs (text)
        article_sp_ids_padded = pad_or_truncate(article_sp_ids + [EOS_ID], max_len, PAD_ID) # Add EOS before pad
        highlights_sp_ids_padded = pad_or_truncate(highlights_sp_ids + [EOS_ID], max_len, PAD_ID)

        # Apply padding/truncation for word-level features (POS, DEP, NER)
        # Note: These are word-level tags. We'll use BERT's word_ids to map them to subwords.
        # So, padding here is based on spaCy token count, then truncate to MAX_LEN.
        # It's okay if this length doesn't exactly match max_len, as BERTDataset will re-align.
        article_pos_ids_padded = pad_or_truncate(current_article_pos_ids_word_level, max_len, PAD_ID)
        article_dep_ids_padded = pad_or_truncate(current_article_dep_ids_word_level, max_len, PAD_ID)
        article_ner_ids_padded = pad_or_truncate(current_article_ner_ids_word_level, max_len, PAD_ID)

        highlights_pos_ids_padded = pad_or_truncate(current_highlights_pos_ids_word_level, max_len, PAD_ID)
        highlights_dep_ids_padded = pad_or_truncate(current_highlights_dep_ids_word_level, max_len, PAD_ID)
        highlights_ner_ids_padded = pad_or_truncate(current_highlights_ner_ids_word_level, max_len, PAD_ID)


        all_articles_ids.append(article_sp_ids_padded)
        all_highlights_ids.append(highlights_sp_ids_padded)
        all_articles_pos_ids.append(article_pos_ids_padded)
        all_highlights_pos_ids.append(highlights_pos_ids_padded)
        all_articles_dep_ids.append(article_dep_ids_padded)
        all_highlights_dep_ids.append(highlights_dep_ids_padded)
        all_articles_ner_tags_word_level.append(article_ner_ids_padded) # Store new word-level tags
        all_highlights_ner_tags_word_level.append(highlights_ner_ids_padded) # Store new word-level tags

    # Convert to PyTorch tensors
    processed_data = {
        'articles_ids': torch.tensor(all_articles_ids, dtype=torch.long),
        'highlights_ids': torch.tensor(all_highlights_ids, dtype=torch.long),
        'articles_pos_ids': torch.tensor(all_articles_pos_ids, dtype=torch.long),
        'highlights_pos_ids': torch.tensor(all_highlights_pos_ids, dtype=torch.long),
        'articles_dep_ids': torch.tensor(all_articles_dep_ids, dtype=torch.long),
        'highlights_dep_ids': torch.tensor(all_highlights_dep_ids, dtype=torch.long),
        'articles_ner_tags_word_level': torch.tensor(all_articles_ner_tags_word_level, dtype=torch.long), # New entry
        'highlights_ner_tags_word_level': torch.tensor(all_highlights_ner_tags_word_level, dtype=torch.long), # New entry
        'pad_id': PAD_ID,
        'bos_id': BOS_ID,
        'eos_id': EOS_ID,
        'vocab_size': sp_tokenizer.get_piece_size(),
        'pos_vocab_size': len(pos_map),
        'dep_vocab_size': len(dep_map),
        'ner_vocab_size': len(ner_map)
    }
    try:
        torch.save(processed_data, output_path)
        print(f"Processed data with spaCy features and word-level NER saved to {output_path}")
    except Exception as e:
        print(f"Error saving processed data to {output_path}: {e}")

if __name__ == "__main__":
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    sp_tokenizer = load_tokenizer(TOKENIZER_MODEL_PATH)

    pos_map, dep_map, ner_map = create_tag_mappings(nlp)
    
    print("POS Tag Mappings:", pos_map)
    print("DEP Tag Mappings:", dep_map)
    print("NER Tag Mappings:", ner_map) # 'O' should be present here now

    train_normalized_path = f"{PROCESSED_DATA_DIR}/train_normalized.csv"
    validation_normalized_path = f"{PROCESSED_DATA_DIR}/validation_normalized.csv"
    test_normalized_path = f"{PROCESSED_DATA_DIR}/test_normalized.csv"

    # process_and_tokenize_data(sp_tokenizer, train_normalized_path,
    #                           f"{PROCESSED_DATA_DIR}/train_processed.pt", MAX_LEN,
    #                           pos_map, dep_map, ner_map)
    # process_and_tokenize_data(sp_tokenizer, validation_normalized_path,
    #                           f"{PROCESSED_DATA_DIR}/validation_processed.pt", MAX_LEN,
    #                           pos_map, dep_map, ner_map)
    process_and_tokenize_data(sp_tokenizer, test_normalized_path,
                              f"{PROCESSED_DATA_DIR}/test_processed.pt", MAX_LEN,
                              pos_map, dep_map, ner_map)

    print("\nPreprocessing complete. Data is saved in .pt files with token IDs and spaCy word-level features.")


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np # For simulating GloVe

# --- Constants for model definition (can be adjusted in main script) ---
# Assuming GloVe 300d embeddings
GLOVE_EMBEDDING_DIM = 300
HIDDEN_DIM = 512 # Wider BiLSTMs
N_LAYERS = 2     # Deeper BiLSTMs
DROPOUT = 0.5

def load_glove_embeddings(vocab_size: int, embedding_dim: int, word_to_idx: dict = None):
    """
    Simulates loading pre-trained GloVe embeddings.
    In a real scenario, you would download and parse glove.6B.300d.txt.
    This function creates a random embedding matrix for demonstration.

    Args:
        vocab_size (int): The size of your model's vocabulary.
        embedding_dim (int): The dimension of the GloVe embeddings (e.g., 300).
        word_to_idx (dict, optional): A dictionary mapping words to their integer IDs.
                                      If provided, actual (or simulated) embeddings will be mapped.
                                      If None, a random matrix is returned for the whole vocab.

    Returns:
        torch.Tensor: An embedding matrix (vocab_size, embedding_dim).
    """
    # For demonstration, we'll create a random embedding matrix.
    # In a real project, you would load actual GloVe embeddings here.
    # Example (conceptual, requires file parsing):
    # embeddings = {}
    # with open('path/to/glove.6B.300d.txt', 'r', encoding='utf-8') as f:
    #     for line in f:
    #         parts = line.split()
    #         word = parts[0]
    #         vector = np.array(parts[1:], dtype=np.float32)
    #         embeddings[word] = vector

    # embedding_matrix = np.random.randn(vocab_size, embedding_dim).astype(np.float32) # Fallback random init
    # for word, idx in word_to_idx.items():
    #     if word in embeddings:
    #         embedding_matrix[idx] = embeddings[word]
    #     else:
    #         # Initialize OOV words randomly or with zeros
    #         embedding_matrix[idx] = np.random.randn(embedding_dim).astype(np.float32) * 0.01

    print(f"Simulating GloVe embedding loading. Creating a random embedding matrix of size ({vocab_size}, {embedding_dim}).")
    embedding_matrix = torch.FloatTensor(vocab_size, embedding_dim).uniform_(-0.25, 0.25) # Random init
    # For PAD_ID, it's common to set its embedding to zeros
    # if word_to_idx and 'PAD_ID' in word_to_idx: # assuming PAD_ID maps to 'PAD_ID' string
    #    embedding_matrix[word_to_idx['PAD_ID']] = 0.0

    return embedding_matrix


class Encoder(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 n_layers: int, dropout: float, pad_idx: int, embedding_matrix: torch.Tensor = None):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=pad_idx)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers,
                           bidirectional=True, dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)

        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor, src_len: torch.Tensor):
        embedded = self.dropout(self.embedding(src))

        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, src_len.cpu(), batch_first=True, enforce_sorted=False)

        packed_outputs, (hidden, cell) = self.rnn(packed_embedded)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True, total_length=src.shape[1])

        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)))
        cell = torch.tanh(self.fc_cell(torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1)))
        hidden = hidden.unsqueeze(0).repeat(self.n_layers, 1, 1)
        cell = cell.unsqueeze(0).repeat(self.n_layers, 1, 1)

        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.attn_hidden = nn.Linear(hidden_dim, hidden_dim)

        self.attn_encoder_outputs = nn.Linear(hidden_dim * 2, hidden_dim)

        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor = None):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1)

        energy = torch.tanh(self.attn_hidden(hidden) + self.attn_encoder_outputs(encoder_outputs))

        attention_scores = self.v(energy).squeeze(2)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e10)

        attention_weights = F.softmax(attention_scores, dim=1)

        context_vector = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context_vector, attention_weights


class PointerGeneratorDecoder(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 n_layers: int, dropout: float, pad_idx: int, embedding_matrix: torch.Tensor = None):
        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=pad_idx)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.dropout = nn.Dropout(dropout)

        self.attention = Attention(hidden_dim)

        self.rnn = nn.LSTM(embedding_dim + (hidden_dim * 2), hidden_dim,
                           num_layers=n_layers, dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)

        # For PGN: Linear layer to compute p_gen
        # Input to p_gen: current decoder hidden state, current context vector, current input embedding
        self.p_gen_linear = nn.Linear(hidden_dim * 3 + embedding_dim, 1) # hidden_dim (s_t) + hidden_dim*2 (c_t) + embedding_dim (y_t_embedding)

        # Output linear layer projects LSTM output to fixed vocabulary size
        self.fc_out = nn.Linear(hidden_dim, vocab_size)


    def forward(self, input_token: torch.Tensor, hidden: torch.Tensor,
                cell: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor,
                src_seq_for_copy: torch.Tensor):
        # input_token = [batch_size, 1]
        # hidden, cell = [n_layers, batch_size, hidden_dim]
        # encoder_outputs = [batch_size, src_len, hidden_dim * 2]
        # mask = [batch_size, src_len]
        # src_seq_for_copy = [batch_size, src_len] (original input token IDs, used for copying)

        embedded = self.dropout(self.embedding(input_token)) # [batch_size, 1, embedding_dim]

        attn_hidden_for_attention = hidden[-1, :, :] # [batch_size, hidden_dim]

        context_vector, attention_weights = self.attention(attn_hidden_for_attention, encoder_outputs, mask)
        # context_vector = [batch_size, hidden_dim * 2]
        # attention_weights = [batch_size, src_len]

        rnn_input = torch.cat((embedded, context_vector.unsqueeze(1)), dim=2) # [batch_size, 1, embedding_dim + hidden_dim * 2]

        output_rnn, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        # output_rnn = [batch_size, 1, hidden_dim]

        # Calculate p_gen
        # Concatenate: output_rnn (squeeze), context_vector, embedded (squeeze)
        p_gen_input = torch.cat((output_rnn.squeeze(1), context_vector, embedded.squeeze(1)), dim=1)
        p_gen = torch.sigmoid(self.p_gen_linear(p_gen_input)) # [batch_size, 1]

        # Calculate P_vocab (distribution over fixed vocabulary)
        P_vocab = F.softmax(self.fc_out(output_rnn.squeeze(1)), dim=1) # [batch_size, vocab_size]

        # Expand attention_weights for the copy mechanism
        # Create a blank distribution for the extended vocabulary.
        # Max extended vocab size: vocab_size + max_src_len (unique OOV words per batch)
        # For simplicity, we'll assume a max_vocab_size that covers all source words in the batch.
        # This part is typically handled in the training loop due to dynamic extended vocabulary sizes.
        # Here, we will return P_vocab, p_gen, and attention_weights.
        # The Seq2SeqPGN will then combine them.

        return P_vocab, p_gen, hidden, cell, attention_weights


class Seq2SeqPGN(nn.Module):
    def __init__(self, encoder: Encoder, decoder: PointerGeneratorDecoder, device: torch.device, pad_idx: int):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.pad_idx = pad_idx

        assert encoder.hidden_dim == decoder.hidden_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Number of layers in encoder and decoder must be equal!"
        assert encoder.embedding.embedding_dim == decoder.embedding.embedding_dim, \
            "Embedding dimensions of encoder and decoder must be equal!"

    def forward(self, src: torch.Tensor, src_len: torch.Tensor, trg: torch.Tensor, trg_len: torch.Tensor,
                src_for_copy: torch.Tensor, teacher_forcing_ratio: float = 0.5):
        
        batch_size = src.shape[0]
        trg_seq_len = trg.shape[1]
        vocab_size = self.decoder.vocab_size

        encoder_outputs, hidden, cell = self.encoder(src, src_len)

        mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
        for j, length in enumerate(src_len):
            mask[j, :length] = True

        # Initialize input to the decoder with the <bos> token
        input_token = trg[:, 0].unsqueeze(1) # [batch_size, 1]

        # Store combined probabilities for loss calculation
        final_dists = []

        for t in range(1, trg_seq_len):
            P_vocab, p_gen, hidden, cell, attention_weights = self.decoder(
                input_token, hidden, cell, encoder_outputs, mask, src_for_copy
            )
            # P_vocab = [batch_size, vocab_size]
            # p_gen = [batch_size, 1]
            # attention_weights = [batch_size, src_len]

            # Combine P_vocab and attention_weights for PGN
            # P_extended = p_gen * P_vocab + (1-p_gen) * P_attention_on_source
            # P_attention_on_source needs to be mapped to the extended vocabulary
            
            # Create a zero tensor for the extended vocabulary distribution
            # Max possible vocabulary size for the batch is vocab_size + max(src_len)
            # A more robust approach might be to find the max token ID in src_for_copy
            # and set extended_vocab_size to max(vocab_size, max_token_id_in_batch + 1)
            
            # For simplicity, we'll assume a max_extended_vocab_size for this forward pass
            # This is a critical part of PGN and needs careful handling.
            # Here, we'll use a fixed maximum extended size based on MAX_LEN + vocab_size
            # A common approach: extended_vocab_size = decoder.vocab_size + max_src_len
            # In practice, max_src_len is fixed by MAX_LEN from preprocessing.
            
            # Get the max token ID across the batch in src_for_copy
            # This accounts for out-of-vocabulary words that are present in the source.
            max_oov_idx = 0
            # Ensure src_for_copy is properly padded/indexed, and max_oov_idx is safe
            # For this simple implementation, we'll assume target vocab covers most words
            # and just handle a fixed max extended vocab size for mapping attention.
            
            # A more accurate way to get extended vocab size:
            # extended_vocab_size = max(vocab_size, torch.max(src_for_copy).item() + 1)
            # This assumes that token IDs from tokenizer are sequential and do not exceed vocab_size
            # for regular words, and OOV words in source might have higher IDs or a separate range.
            # In sentencepiece, OOV words are broken down, so the IDs are generally within vocab_size.
            # So the PGN copy mechanism usually works on the attention weights directly.

            # Let's consider the simplest form of PGN output combination, where
            # the attention weights are distributed onto the vocabulary directly.
            # This requires knowing which source token corresponds to which vocab ID.
            # The `src_for_copy` tensor contains the original token IDs of the source sequence.
            # We will scatter the attention_weights onto a tensor of `vocab_size` + `max_src_len`
            # For each batch item, scatter attention_weights (which are over source positions)
            # onto the corresponding word IDs in the extended vocabulary.

            # Create a zero tensor to hold the final distribution over the extended vocabulary
            # The size will be max(actual_vocab_size, max_token_id_in_source_batch + 1)
            # Let's dynamically get the max ID from source within this batch
            # This is important if your tokenizer's IDs for words are not simply 0 to vocab_size-1
            # or if you are using a separate mechanism for OOV.
            
            # For this implementation, we will use `decoder.vocab_size` for `P_vocab`
            # and `src_for_copy` (actual token IDs) for the attention distribution.
            
            # P_vocab is already [batch_size, vocab_size]
            
            # attention_weights_on_src_tokens = [batch_size, src_len]
            # We need to distribute these attention weights onto the vocabulary IDs present in src_for_copy.
            # This requires creating an extended vocabulary distribution.
            
            # Find max token ID in source for the batch to determine extended_vocab_size
            # Assuming src_for_copy contains the actual token IDs (including potential OOV mapped to high IDs)
            # Or, if OOV words are mapped to a specific ID range, use that.
            # For sentencepiece, all IDs are within vocab_size. So, we scatter on vocab_size.
            
            # Maximum index for scattering. We will use the original vocab_size + max source sequence length
            # to accommodate potentially new words. This is a common heuristic.
            # A more precise way would involve tracking OOV words during preprocessing.
            # For now, let's assume `vocab_size` is the max index in `P_vocab`.
            
            # Initialize extended_vocab_dists with zeros
            # The target_token_ids in src_for_copy can go up to vocab_size - 1 for in-vocab words.
            # For PGN, we extend the vocabulary to include *all unique words in the source batch*.
            # This is usually done by creating an `extended_vocab_size` that is `base_vocab_size + num_unique_oov_in_batch`.
            # For simplicity, and since SentencePiece handles OOV by subwords, we'll directly scatter
            # attention weights onto the `vocab_size` dimension, handling the standard vocabulary.
            # This means `P_attention_on_source` will be of size `[batch_size, vocab_size]`.

            # Create `P_attention_on_source` by scattering `attention_weights`
            # onto `src_for_copy` indices. This assumes `src_for_copy` contains valid vocabulary IDs.
            # If src_for_copy contains out-of-vocabulary words not in the decoder's vocab_size,
            # this needs special handling (e.g., dynamic extended vocabulary).
            
            # For now, let's use the simplest approach where we scatter attention weights
            # to the vocabulary indices in `src_for_copy`. If `src_for_copy` has values > vocab_size,
            # this scattering will extend the tensor, which is what we want for PGN.
            
            max_src_token_id = src_for_copy.max().item()
            extended_vocab_size = max(vocab_size, max_src_token_id + 1)
            
            # Create a zero tensor for the extended vocabulary distribution for attention
            P_attn_on_src_tokens = torch.zeros(batch_size, extended_vocab_size).to(self.device)
            # Scatter attention_weights (which are over source positions) to the vocabulary indices
            # provided by src_for_copy.
            # Add a small value to attention_weights before scattering to handle zero attention for padded tokens
            # For padding, attention_weights should be 0.
            
            # Ensure src_for_copy values are valid indices for scattering
            # clamp src_for_copy values to be within [0, extended_vocab_size - 1]
            clamped_src_for_copy = src_for_copy.clamp(min=0, max=extended_vocab_size - 1)
            
            P_attn_on_src_tokens.scatter_add_(dim=1, index=clamped_src_for_copy, src=attention_weights)
            # P_attn_on_src_tokens = [batch_size, extended_vocab_size]

            # Combine P_vocab and P_attn_on_src_tokens using p_gen
            # Ensure P_vocab is extended to extended_vocab_size if smaller
            P_vocab_extended = torch.zeros(batch_size, extended_vocab_size).to(self.device)
            P_vocab_extended[:, :vocab_size] = P_vocab

            final_dist = p_gen * P_vocab_extended + (1 - p_gen) * P_attn_on_src_tokens
            final_dists.append(final_dist)
            
            # For next input, use teacher forcing or sampled token
            # Sampled token needs to be drawn from the *extended* distribution if using PGN properly
            # However, for simplicity and to match the standard training loop (which uses fixed vocab output),
            # we will sample from P_vocab when teacher forcing is off.
            # If you want to sample from the full PGN distribution, the target 'trg' needs to be extended.
            # For scheduled sampling, it's typical to sample from P_vocab.

            # Decide whether to use teacher forcing or not
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            # Get the token with the highest probability from the model's P_vocab output
            # For training stability, usually sample from P_vocab, not the full PGN dist
            top1 = P_vocab.argmax(1) # [batch_size]

            input_token = trg[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1) # [batch_size, 1]

        return torch.stack(final_dists, dim=1) # [batch_size, trg_seq_len - 1, extended_vocab_size]


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np # For simulating GloVe

# --- Constants for model definition (can be adjusted in main script) ---
# Re-using constants from previous models for consistency
GLOVE_EMBEDDING_DIM = 300 # Using 300d for GloVe if available, otherwise 100
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5
FILTER_SIZES = [3, 4, 5] # CNN filter sizes (n-grams)
NUM_FILTERS = 100       # Number of filters per size

def load_glove_embeddings(vocab_size: int, embedding_dim: int, word_to_idx: dict = None):
    """
    Simulates loading pre-trained GloVe embeddings.
    In a real scenario, you would download and parse glove.6B.300d.txt.
    This function creates a random embedding matrix for demonstration.
    """
    print(f"Simulating GloVe embedding loading. Creating a random embedding matrix of size ({vocab_size}, {embedding_dim}).")
    embedding_matrix = torch.FloatTensor(vocab_size, embedding_dim).uniform_(-0.25, 0.25)
    return embedding_matrix


class BiLSTMCNNEncoder(nn.Module):
    """
    Encoder for abstractive summarization.
    Combines CNN for local feature extraction with BiLSTM for sequential understanding.
    CNN processes input embeddings to create enhanced features that feed into the BiLSTM.
    """
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 n_layers: int, dropout: float, pad_idx: int,
                 filter_sizes: list[int], num_filters: int,
                 embedding_matrix: torch.Tensor = None):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.num_filters = num_filters
        self.filter_sizes = filter_sizes

        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=pad_idx)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # CNN Layers: Apply Conv1d for different filter sizes
        # Conv1d expects input in shape (batch_size, channels, sequence_length)
        # Our embedded input is (batch_size, sequence_length, embedding_dim)
        # We will apply CNNs to the embedded input, and concatenate their outputs
        # to form an "enhanced" embedding that feeds into the BiLSTM.
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, # Input channels are embedding_dim
                      out_channels=num_filters,  # Output channels are num_filters
                      kernel_size=fs,            # Kernel size
                      padding=fs // 2)           # Add padding to maintain sequence length
            for fs in filter_sizes
        ])

        # Linear layer to project concatenated CNN outputs back to embedding_dim
        # or a new dimension suitable for LSTM input.
        # Output of CNNs for each token will be (num_filters * len(filter_sizes))
        self.cnn_output_dim = num_filters * len(filter_sizes)
        self.fc_cnn_to_lstm = nn.Linear(self.cnn_output_dim, embedding_dim) # Project back to embedding_dim

        # BiLSTM layer: Processes the CNN-enhanced features
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers,
                           bidirectional=True, dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)

        # Linear layers to project concatenated BiLSTM hidden/cell states for decoder initialization
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor, src_len: torch.Tensor):
        # src = [batch size, src len]
        # src_len = [batch size]

        embedded = self.dropout(self.embedding(src))
        # embedded = [batch size, src len, embedding dim]

        # --- CNN Feature Extraction ---
        # Permute for Conv1d: [batch size, embedding dim, src len]
        embedded_for_cnn = embedded.permute(0, 2, 1)

        # Apply convolutions and concatenate outputs
        cnn_features_per_token = []
        for conv in self.convs:
            # conv_output = [batch size, num_filters, src len] due to padding
            cnn_output = F.relu(conv(embedded_for_cnn))
            cnn_features_per_token.append(cnn_output)
        
        # Concatenate outputs from all CNNs along the channel dimension
        # result: [batch size, num_filters * len(filter_sizes), src len]
        combined_cnn_channels = torch.cat(cnn_features_per_token, dim=1)

        # Permute back to [batch size, src len, num_filters * len(filter_sizes)]
        combined_cnn_features = combined_cnn_channels.permute(0, 2, 1)
        
        # Project to the desired LSTM input dimension (e.g., original embedding_dim)
        lstm_input = self.fc_cnn_to_lstm(self.dropout(combined_cnn_features))
        # lstm_input = [batch size, src len, embedding dim]


        # --- BiLSTM Processing ---
        packed_lstm_input = nn.utils.rnn.pack_padded_sequence(lstm_input, src_len.cpu(), batch_first=True, enforce_sorted=False)

        packed_outputs, (hidden, cell) = self.lstm(packed_lstm_input)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True, total_length=src.shape[1])
        # outputs = [batch size, src len, hidden dim * 2] (concatenated forward and backward LSTM outputs)

        # Initial hidden/cell states for decoder (from last encoder hidden/cell states)
        # Concatenate forward and backward last hidden/cell states
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)))
        cell = torch.tanh(self.fc_cell(torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1)))
        
        # Repeat for decoder's n_layers (decoder is often unidirectional, so need 1 layer's hidden state repeated)
        hidden = hidden.unsqueeze(0).repeat(self.n_layers, 1, 1)
        cell = cell.unsqueeze(0).repeat(self.n_layers, 1, 1)

        return outputs, hidden, cell


class Attention(nn.Module):
    """
    Bahdanau-style Attention Mechanism.
    Reused from model/seq2seq_pgn_model.py
    """
    def __init__(self, hidden_dim: int):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.attn_hidden = nn.Linear(hidden_dim, hidden_dim) # For decoder hidden state
        self.attn_encoder_outputs = nn.Linear(hidden_dim * 2, hidden_dim) # For encoder outputs
        self.v = nn.Linear(hidden_dim, 1, bias=False) # For combining energy scores

    def forward(self, hidden: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor = None):
        # hidden = [batch_size, hidden_dim] (last hidden state of decoder)
        # encoder_outputs = [batch_size, src_len, hidden_dim * 2]
        # mask = [batch_size, src_len] (1 for real tokens, 0 for padding)

        src_len = encoder_outputs.shape[1]

        # Reshape decoder hidden state to broadcast across src_len
        # [batch_size, 1, hidden_dim]
        hidden_for_attention = hidden.unsqueeze(1)

        # Calculate energy: score(h_t, s_i) = v^T * tanh(W_h * h_t + W_s * s_i)
        energy = torch.tanh(self.attn_hidden(hidden_for_attention) + self.attn_encoder_outputs(encoder_outputs))
        # energy = [batch_size, src_len, hidden_dim]

        attention_scores = self.v(energy).squeeze(2)
        # attention_scores = [batch_size, src_len]

        if mask is not None:
            # Apply mask to attention scores: fill padded values with -infinity
            attention_scores = attention_scores.masked_fill(mask == 0, -1e10)

        attention_weights = F.softmax(attention_scores, dim=1)
        # attention_weights = [batch_size, src_len]

        # Calculate context vector: weighted sum of encoder outputs
        context_vector = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        # context_vector = [batch_size, hidden_dim * 2]

        return context_vector, attention_weights


class BiLSTMCNNPGNDecoder(nn.Module):
    """
    Decoder for abstractive summarization with Pointer-Generator Network.
    Reuses PGN logic from model/seq2seq_pgn_model.py
    """
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 n_layers: int, dropout: float, pad_idx: int,
                 encoder_output_dim: int): # hidden_dim * 2 for BiLSTM encoder
        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.encoder_output_dim = encoder_output_dim

        # Decoder embedding layer (often separate from encoder)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.dropout = nn.Dropout(dropout)

        self.attention = Attention(hidden_dim) # Attention module

        # Decoder LSTM: input is embedded token + context vector from attention
        self.rnn = nn.LSTM(embedding_dim + encoder_output_dim, hidden_dim,
                           num_layers=n_layers, dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)

        # For PGN: Linear layer to compute p_gen
        # Input to p_gen: current decoder hidden state (s_t), current context vector (c_t), current input embedding (y_t_embedding)
        self.p_gen_linear = nn.Linear(hidden_dim + encoder_output_dim + embedding_dim, 1)

        # Output linear layer projects LSTM output to fixed vocabulary size
        self.fc_out = nn.Linear(hidden_dim, vocab_size)


    def forward(self, input_token: torch.Tensor, hidden: torch.Tensor,
                cell: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor,
                src_seq_for_copy: torch.Tensor):
        # input_token = [batch_size, 1] (previous generated token or ground truth)
        # hidden, cell = [n_layers, batch_size, hidden_dim]
        # encoder_outputs = [batch_size, src_len, encoder_output_dim]
        # mask = [batch_size, src_len]
        # src_seq_for_copy = [batch_size, src_len] (original input token IDs, used for copying)

        embedded = self.dropout(self.embedding(input_token)) # [batch_size, 1, embedding_dim]

        # Use the top layer's hidden state for attention computation
        attn_hidden_for_attention = hidden[-1, :, :] # [batch_size, hidden_dim]

        context_vector, attention_weights = self.attention(attn_hidden_for_attention, encoder_outputs, mask)
        # context_vector = [batch_size, encoder_output_dim]
        # attention_weights = [batch_size, src_len] (alpha_t, attention distribution over source)

        # Input to decoder LSTM: concatenated embedded token and context vector
        rnn_input = torch.cat((embedded, context_vector.unsqueeze(1)), dim=2)
        # rnn_input = [batch_size, 1, embedding_dim + encoder_output_dim]

        output_rnn, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        # output_rnn = [batch_size, 1, hidden_dim]

        # Calculate p_gen (probability of generating from vocabulary vs. copying from source)
        # Input to p_gen: decoder hidden state (s_t), context vector (c_t), input embedding (y_t_embedding)
        p_gen_input = torch.cat((output_rnn.squeeze(1), context_vector, embedded.squeeze(1)), dim=1)
        p_gen = torch.sigmoid(self.p_gen_linear(p_gen_input)) # [batch_size, 1]

        # Calculate P_vocab (distribution over fixed vocabulary)
        P_vocab = F.softmax(self.fc_out(output_rnn.squeeze(1)), dim=1) # [batch_size, vocab_size]

        return P_vocab, p_gen, hidden, cell, attention_weights


class Seq2SeqBiLSTMCNNPGN(nn.Module):
    """
    Full Encoder-Decoder Sequence-to-Sequence model with BiLSTM-CNN Encoder,
    Attention, and Pointer-Generator Network Decoder.
    """
    def __init__(self, encoder: BiLSTMCNNEncoder, decoder: BiLSTMCNNPGNDecoder, device: torch.device, pad_idx: int):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.pad_idx = pad_idx

        # Ensure hidden dimensions and layers match for compatibility
        assert encoder.hidden_dim == decoder.hidden_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Number of layers in encoder and decoder must be equal!"
        # Decoder's embedding dim might be different from encoder's original embedding_dim,
        # but the encoder's input to its LSTM is projected to embedding_dim.
        # We need to ensure the decoder's LSTM input dimension matches the encoder's output.

    def forward(self, src: torch.Tensor, src_len: torch.Tensor, trg: torch.Tensor, trg_len: torch.Tensor,
                src_for_copy: torch.Tensor, teacher_forcing_ratio: float = 0.5):
        
        batch_size = src.shape[0]
        trg_seq_len = trg.shape[1]
        vocab_size = self.decoder.vocab_size

        # Encoder forward pass
        encoder_outputs, hidden, cell = self.encoder(src, src_len)
        # encoder_outputs = [batch_size, src len, encoder_output_dim (hidden_dim * 2)]
        # hidden, cell = [n_layers, batch_size, hidden_dim]

        # Create mask for encoder outputs (for attention)
        mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
        for j, length in enumerate(src_len):
            mask[j, :length] = True

        # Initialize input to the decoder with the <bos> token
        input_token = trg[:, 0].unsqueeze(1) # [batch_size, 1] (<bos> token)

        # Store combined probabilities for loss calculation
        final_dists = []

        for t in range(1, trg_seq_len): # Iterate from 1 to trg_seq_len-1 (predicting actual tokens)
            P_vocab, p_gen, hidden, cell, attention_weights = self.decoder(
                input_token, hidden, cell, encoder_outputs, mask, src_for_copy
            )
            # P_vocab = [batch_size, vocab_size]
            # p_gen = [batch_size, 1]
            # attention_weights = [batch_size, src_len]

            # --- PGN: Combine P_vocab and attention_weights for P_extended ---
            # Determine the maximum index needed for the extended vocabulary.
            # This is max(base_vocab_size, max_token_id_in_source + 1).
            # `src_for_copy` contains the actual token IDs from the input article.
            max_src_token_id = src_for_copy.max().item() if src_for_copy.numel() > 0 else 0
            extended_vocab_size = max(vocab_size, max_src_token_id + 1)
            
            # Create P_vocab_extended by padding P_vocab to extended_vocab_size
            P_vocab_extended = torch.zeros(batch_size, extended_vocab_size).to(self.device)
            P_vocab_extended[:, :vocab_size] = P_vocab

            # Create P_attn_on_src_tokens by scattering attention weights onto source token IDs
            P_attn_on_src_tokens = torch.zeros(batch_size, extended_vocab_size).to(self.device)
            # Clamp src_for_copy values to be within [0, extended_vocab_size - 1]
            # This handles cases where src_for_copy might contain IDs that exceed the
            # dynamically calculated extended_vocab_size (e.g., if max_src_token_id is less
            # than vocab_size due to specific batch content, but other OOV are possible).
            # However, for SentencePiece, source IDs are typically within vocab_size.
            clamped_src_for_copy = src_for_copy.clamp(min=0, max=extended_vocab_size - 1)
            P_attn_on_src_tokens.scatter_add_(dim=1, index=clamped_src_for_copy, src=attention_weights)
            
            # Final distribution: p_gen * P_vocab + (1 - p_gen) * P_attention_on_source
            final_dist = p_gen * P_vocab_extended + (1 - p_gen) * P_attn_on_src_tokens
            final_dists.append(final_dist)
            
            # --- Scheduled Sampling ---
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            # For next input, sample from P_vocab for simplicity in scheduled sampling
            # (as the target `trg` only contains base vocabulary IDs)
            top1 = P_vocab.argmax(1) # [batch_size]

            input_token = trg[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)

        # Return concatenated final distributions for all predicted timesteps
        return torch.stack(final_dists, dim=1) # [batch_size, trg_seq_len - 1, extended_vocab_size]


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import math # For scheduled sampling
import sentencepiece as spm # For tokenizer info

# --- Constants for training (match model definition) ---
GLOVE_EMBEDDING_DIM = 300
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5
FILTER_SIZES = [3, 5] # Changed from [3, 4, 5] to ensure consistent output lengths from CNN
NUM_FILTERS = 100

BATCH_SIZE = 32
NUM_EPOCHS = 10
CLIP_GRADIENT = 1.0
LEARNING_RATE = 1e-4 # Often lower for models with pre-trained embeddings
SCHEDULED_SAMPLING_DECAY = 0.995 # Decay rate for teacher forcing ratio per batch

BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints_bilstm_cnn_pgn" # New checkpoint directory
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDataset(Dataset):
    """
    Dataset for abstractive summarization.
    Loads preprocessed data (token IDs, highlights IDs, etc.) from a PyTorch .pt file.
    """
    def __init__(self, data_path: str):
        print(f"Loading data from {data_path} for BiLSTM-CNN PGN training...")
        try:
            data = torch.load(data_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Processed data file not found at {data_path}")
        except Exception as e:
            raise Exception(f"Error loading data from {data_path}: {e}")

        self.articles = data['articles_ids']
        self.highlights = data['highlights_ids']
        self.pad_id = data['pad_id']
        self.bos_id = data['bos_id']
        self.eos_id = data['eos_id']
        self.vocab_size = data['vocab_size']
        self.max_len_article = self.articles.shape[1]
        self.max_len_highlight = self.highlights.shape[1]


        self.article_lengths = self._calculate_lengths(self.articles, self.pad_id)
        # Highlight lengths should include EOS as it's the target sequence end marker
        self.highlight_lengths = self._calculate_lengths(self.highlights, self.eos_id, include_eos=True)

        print(f"Loaded {len(self.articles)} samples.")
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"PAD ID: {self.pad_id}, BOS ID: {self.bos_id}, EOS ID: {self.eos_id}")
        print(f"Max article length: {self.max_len_article}, Max highlight length: {self.max_len_highlight}")


    def _calculate_lengths(self, sequences: torch.Tensor, stop_id: int, include_eos: bool = False) -> torch.Tensor:
        lengths = []
        for seq in sequences:
            # Find the index of the first stop_id (PAD or EOS)
            indices = (seq == stop_id).nonzero(as_tuple=True)[0]
            if len(indices) > 0:
                length = indices[0].item() + (1 if include_eos else 0)
            else:
                length = len(seq) # If stop_id not found, take full length
            lengths.append(length)
        return torch.tensor(lengths, dtype=torch.long)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        return {
            'article': self.articles[idx],
            'highlight': self.highlights[idx],
            'article_len': self.article_lengths[idx],
            'highlight_len': self.highlight_lengths[idx]
        }

def pgn_loss_fn(final_dists, target, pad_idx, vocab_size):
    """
    Custom loss function for PGN.
    It expects final_dists to be [batch_size, trg_seq_len-1, extended_vocab_size]
    and target to be [batch_size, trg_seq_len-1].
    The target indices can go beyond the base vocab_size for OOV words.
    """
    batch_size, trg_seq_len_minus_1, extended_vocab_size = final_dists.shape
    
    # Flatten the distributions and targets for CrossEntropyLoss (NLLLoss here)
    final_dists_flat = final_dists.reshape(-1, extended_vocab_size)
    target_flat = target.reshape(-1)

    log_final_dists = torch.log(final_dists_flat + 1e-12) # Add small epsilon for numerical stability

    # Create a mask for valid (non-padded) target tokens
    non_pad_mask = (target_flat != pad_idx)

    # Apply the mask
    log_final_dists_masked = log_final_dists[non_pad_mask]
    target_flat_masked = target_flat[non_pad_mask]

    if target_flat_masked.numel() == 0:
        return torch.tensor(0.0, device=final_dists.device, requires_grad=True)

    # Ensure target_flat_masked indices are within the bounds of extended_vocab_size
    valid_target_indices_in_ext_vocab = (target_flat_masked < extended_vocab_size)
    
    log_final_dists_valid_targets = log_final_dists_masked[valid_target_indices_in_ext_vocab]
    target_flat_valid_targets = target_flat_masked[valid_target_indices_in_ext_vocab]

    if target_flat_valid_targets.numel() == 0:
        return torch.tensor(0.0, device=final_dists.device, requires_grad=True)

    # Use NLLLoss since we have log probabilities
    loss = F.nll_loss(log_final_dists_valid_targets, target_flat_valid_targets, reduction='sum')
    
    # Normalize by the number of non-padded tokens
    loss = loss / target_flat_valid_targets.numel()

    return loss

def train_model_bilstm_cnn_pgn(model, dataloader, optimizer, pad_idx, eos_id, clip_gradient, device,
                                initial_teacher_forcing_ratio: float = 1.0, decay_rate: float = 0.995):
    
    model.train()
    epoch_loss = 0
    current_teacher_forcing_ratio = initial_teacher_forcing_ratio

    pbar = tqdm(dataloader, desc="Training BiLSTM-CNN PGN")

    for i, batch in enumerate(pbar):
        src = batch['article'].to(device)
        trg = batch['highlight'].to(device)
        src_len = batch['article_len'].to(device)
        trg_len = batch['highlight_len'].to(device)

        # src_for_copy is the article IDs (for PGN copy mechanism)
        src_for_copy = src

        optimizer.zero_grad()

        # Forward pass with scheduled sampling
        final_dists = model(src, src_len, trg, trg_len, src_for_copy, current_teacher_forcing_ratio)
        # final_dists = [batch_size, trg_seq_len-1, extended_vocab_size]

        # Target for loss is trg[:, 1:] because the model predicts the (t+1)th token
        # given the t-th token, so first token of trg (BOS) is not predicted.
        loss = pgn_loss_fn(final_dists, trg[:, 1:], pad_idx, model.decoder.vocab_size)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_gradient)

        optimizer.step()

        epoch_loss += loss.item()
        
        # Update teacher forcing ratio
        current_teacher_forcing_ratio *= decay_rate
        pbar.set_postfix(loss=loss.item(), tf_ratio=f"{current_teacher_forcing_ratio:.4f}")

    return epoch_loss / len(dataloader), current_teacher_forcing_ratio

def evaluate_model_bilstm_cnn_pgn(model, dataloader, pad_idx, eos_id, device):
    model.eval()
    epoch_loss = 0
    
    pbar = tqdm(dataloader, desc="Validation BiLSTM-CNN PGN")

    with torch.no_grad():
        for i, batch in enumerate(pbar):
            src = batch['article'].to(device)
            trg = batch['highlight'].to(device)
            src_len = batch['article_len'].to(device)
            trg_len = batch['highlight_len'].to(device)

            src_for_copy = src

            # No teacher forcing during evaluation (ratio = 0.0)
            final_dists = model(src, src_len, trg, trg_len, src_for_copy, 0.0)
            
            loss = pgn_loss_fn(final_dists, trg[:, 1:], pad_idx, model.decoder.vocab_size)

            epoch_loss += loss.item()
            pbar.set_postfix(val_loss=loss.item())

    return epoch_loss / len(dataloader)


if __name__ == "__main__":
    train_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/train_processed.pt")
    val_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/validation_processed.pt")

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    VOCAB_SIZE = train_dataset.vocab_size
    PAD_ID = train_dataset.pad_id
    BOS_ID = train_dataset.bos_id
    EOS_ID = train_dataset.eos_id
    MAX_LEN_ARTICLE = train_dataset.max_len_article # Get actual max lengths from data

    # Load simulated GloVe embeddings
    sp_tokenizer_path = f"{BASE_DIR}/tokenizer.model"
    if not os.path.exists(sp_tokenizer_path):
        print(f"Error: SentencePiece tokenizer model not found at {sp_tokenizer_path}.")
        print("Please ensure your tokenizer is trained and saved.")
        exit()
    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer.load(sp_tokenizer_path)
    word_to_idx = {sp_tokenizer.id_to_piece(i): i for i in range(VOCAB_SIZE)}

    glove_embedding_matrix = load_glove_embeddings(VOCAB_SIZE, GLOVE_EMBEDDING_DIM, word_to_idx=word_to_idx)
    
    encoder = BiLSTMCNNEncoder(VOCAB_SIZE, GLOVE_EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID,
                               filter_sizes=FILTER_SIZES, num_filters=NUM_FILTERS,
                               embedding_matrix=glove_embedding_matrix).to(DEVICE)
    
    decoder = BiLSTMCNNPGNDecoder(VOCAB_SIZE, GLOVE_EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID,
                                  encoder_output_dim=HIDDEN_DIM * 2).to(DEVICE) # Encoder output is 2 * hidden_dim
    
    model = Seq2SeqBiLSTMCNNPGN(encoder, decoder, DEVICE, PAD_ID).to(DEVICE)

    def init_weights(m):
        for name, param in m.named_parameters():
            if 'weight' in name and len(param.shape) > 1: # Only for weights of linear/LSTM, not biases or 1D
                nn.init.xavier_uniform_(param.data)
            elif 'bias' in name:
                nn.init.constant_(param.data, 0)
    model.apply(init_weights)

    print(f"Number of parameters in BiLSTM-CNN PGN model: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float('inf')
    current_teacher_forcing_ratio = 1.0 # Start with full teacher forcing

    print("\nStarting BiLSTM-CNN PGN Training...")
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

        train_loss, current_teacher_forcing_ratio = train_model_bilstm_cnn_pgn(
            model, train_dataloader, optimizer, PAD_ID, EOS_ID, CLIP_GRADIENT, DEVICE,
            initial_teacher_forcing_ratio=current_teacher_forcing_ratio, decay_rate=SCHEDULED_SAMPLING_DECAY
        )
        val_loss = evaluate_model_bilstm_cnn_pgn(model, val_dataloader, PAD_ID, EOS_ID, DEVICE)

        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'encoder_state_dict': model.encoder.state_dict(),
                'decoder_state_dict': model.decoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'vocab_size': VOCAB_SIZE,
                'pad_id': PAD_ID,
                'bos_id': BOS_ID,
                'eos_id': EOS_ID,
                'max_len_article': MAX_LEN_ARTICLE, # Save max article length
                'embedding_dim': GLOVE_EMBEDDING_DIM,
                'hidden_dim': HIDDEN_DIM,
                'n_layers': N_LAYERS,
                'dropout': DROPOUT,
                'filter_sizes': FILTER_SIZES,
                'num_filters': NUM_FILTERS
            }, f"{CHECKPOINTS_DIR}/bilstm_cnn_pgn_best_model.pt")
            print(f"  Saved best BiLSTM-CNN PGN model with validation loss: {best_val_loss:.4f}")

    print("\nBiLSTM-CNN PGN training complete!")


Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/train_processed.pt for BiLSTM-CNN PGN training...
Loaded 20000 samples.
Vocabulary size: 30000
PAD ID: 0, BOS ID: 2, EOS ID: 3
Max article length: 64, Max highlight length: 64
Loading data from /kaggle/working/TextSummarization/data/processed/validation_processed.pt for BiLSTM-CNN PGN training...
Loaded 13368 samples.
Vocabulary size: 30000
PAD ID: 0, BOS ID: 2, EOS ID: 3
Max article length: 64, Max highlight length: 64


NameError: name 'load_glove_embeddings' is not defined

In [7]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import json
import re
import glob
import sentencepiece as spm # Import SentencePiece

import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.corpus import wordnet
import textstat

try:
    nltk.data.find('corpora/wordnet')
except (LookupError, Exception):
    print("Downloading 'wordnet' NLTK data...")
    nltk.download('wordnet')
try:
    nltk.data.find('tokenizers/punkt')
except (LookupError, Exception):
    print("Downloading 'punkt' NLTK data...")
    nltk.download('punkt')

from rouge_score import rouge_scorer
from bert_score import score as bert_score_calc


# --- Constants for evaluation (should match training and model definition) ---
GLOVE_EMBEDDING_DIM = 300
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5 # Dropout is usually 0 during evaluation, but kept for model definition consistency
FILTER_SIZES = [3, 5] # Corrected to match the training script's filter sizes
NUM_FILTERS = 100

# MAX_LEN_HIGHLIGHT: Should be determined from your preprocessed data, or set as a fixed max for generation.
# For evaluation, we typically generate up to a certain length or until EOS.
MAX_GENERATION_LENGTH = 100 # Maximum length for generated summaries during evaluation

BATCH_SIZE_EVAL = 64

BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints_bilstm_cnn_pgn" # New checkpoint directory
RESULTS_DIR = f"{BASE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDataset(Dataset):
    def __init__(self, data_path: str):
        print(f"Loading data from {data_path} for BiLSTM-CNN PGN evaluation...")
        try:
            data = torch.load(data_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Processed data file not found at {data_path}")
        except Exception as e:
            raise Exception(f"Error loading data from {data_path}: {e}")

        self.articles = data['articles_ids']
        self.highlights = data['highlights_ids'] # Reference highlights
        self.pad_id = data['pad_id']
        self.bos_id = data['bos_id']
        self.eos_id = data['eos_id']
        self.vocab_size = data['vocab_size']
        self.max_len_article = self.articles.shape[1]
        self.max_len_highlight = self.highlights.shape[1] # For reference length

        self.article_lengths = self._calculate_lengths(self.articles, self.pad_id)
        # Highlight lengths are mainly for consistency, not directly used for generation in this class
        self.highlight_lengths = self._calculate_lengths(self.highlights, self.eos_id, include_eos=True)
        
        print(f"Loaded {len(self.articles)} samples for evaluation.")

    def _calculate_lengths(self, sequences: torch.Tensor, stop_id: int, include_eos: bool = False) -> torch.Tensor:
        lengths = []
        for seq in sequences:
            indices = (seq == stop_id).nonzero(as_tuple=True)[0]
            if len(indices) > 0:
                length = indices[0].item() + (1 if include_eos else 0)
            else:
                length = len(seq)
            lengths.append(length)
        return torch.tensor(lengths, dtype=torch.long)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        return {
            'article': self.articles[idx],
            'highlight': self.highlights[idx], # Reference highlight
            'article_len': self.article_lengths[idx],
            'highlight_len': self.highlight_lengths[idx]
        }

class BiLSTMCNNPGNGenerator(nn.Module):
    """
    Generator class for BiLSTM-CNN PGN model to generate summaries during evaluation.
    Performs greedy decoding.
    """
    def __init__(self, model: Seq2SeqBiLSTMCNNPGN, bos_id: int, eos_id: int, max_len_generate: int, device: torch.device):
        super().__init__()
        self.model = model
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.max_len_generate = max_len_generate
        self.device = device

    def generate(self, src: torch.Tensor, src_len: torch.Tensor, src_for_copy: torch.Tensor) -> torch.Tensor:
        self.model.eval() # Set model to evaluation mode

        with torch.no_grad():
            batch_size = src.shape[0]

            # Encoder forward pass
            encoder_outputs, hidden, cell = self.model.encoder(src, src_len)

            # Create mask for attention
            mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
            for j, length in enumerate(src_len):
                mask[j, :length] = True

            # Initialize input to the decoder with the <bos> token
            input_token = torch.full((batch_size, 1), self.bos_id, dtype=torch.long).to(self.device)

            # Tensor to store generated sequence IDs
            generated_sequences = torch.zeros(batch_size, self.max_len_generate, dtype=torch.long).to(self.device)
            
            # Keep track of finished sequences
            finished_sequences = torch.zeros(batch_size, dtype=torch.bool).to(self.device) # True if sequence has generated EOS

            for t in range(self.max_len_generate):
                # Decoder forward pass during generation (no teacher forcing)
                P_vocab, p_gen, hidden, cell, attention_weights = self.model.decoder(
                    input_token, hidden, cell, encoder_outputs, mask, src_for_copy
                )
                
                # --- PGN: Combine P_vocab and attention_weights for P_extended ---
                vocab_size = self.model.decoder.vocab_size
                max_src_token_id = src_for_copy.max().item() if src_for_copy.numel() > 0 else 0
                extended_vocab_size = max(vocab_size, max_src_token_id + 1)
                
                P_vocab_extended = torch.zeros(batch_size, extended_vocab_size).to(self.device)
                P_vocab_extended[:, :vocab_size] = P_vocab

                P_attn_on_src_tokens = torch.zeros(batch_size, extended_vocab_size).to(self.device)
                clamped_src_for_copy = src_for_copy.clamp(min=0, max=extended_vocab_size - 1)
                P_attn_on_src_tokens.scatter_add_(dim=1, index=clamped_src_for_copy, src=attention_weights)
                
                final_dist = p_gen * P_vocab_extended + (1 - p_gen) * P_attn_on_src_tokens
                
                # Select the next token greedily
                predicted_token = final_dist.argmax(1) # [batch_size]

                # Store the predicted token
                # Only update if sequence is not yet finished
                generated_sequences[:, t] = torch.where(
                    finished_sequences,
                    self.eos_id, # If finished, keep outputting EOS or PAD
                    predicted_token
                )
                
                # Update finished sequences: if EOS was predicted, mark as finished
                finished_sequences = finished_sequences | (predicted_token == self.eos_id)

                input_token = predicted_token.unsqueeze(1) # Use the newly predicted token as input for next step

                # If all sequences are finished, break early
                if finished_sequences.all():
                    break
            
            return generated_sequences

def convert_ids_to_text(sp_tokenizer, token_ids: torch.Tensor, eos_id: int, pad_id: int) -> list[str]:
    texts = []
    for seq_ids in token_ids.tolist():
        tokens = []
        for tok_id in seq_ids:
            if tok_id == eos_id: # Stop at EOS
                break
            if tok_id == pad_id: # Ignore PAD
                continue
            # Handle SentencePiece prefixes (e.g., ' ' for word starts)
            piece = sp_tokenizer.id_to_piece(tok_id)
            tokens.append(piece)

        # Join SentencePiece pieces and replace ' ' with actual spaces
        text = "".join(tokens).replace(" ", " ").strip()
        texts.append(text)
    return texts

def calculate_cohesion_jaccard(text: str) -> float:
    sentences = nltk.sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0

    scores = []
    for i in range(len(sentences) - 1):
        words1 = set(nltk.word_tokenize(sentences[i].lower()))
        words2 = set(nltk.word_tokenize(sentences[i+1].lower()))
        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))
        if union > 0:
            scores.append(intersection / union)
        else:
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0


def evaluate_model_bilstm_cnn_pgn(generator: BiLSTMCNNPGNGenerator, dataloader: DataLoader, sp_tokenizer,
                                   pad_id: int, eos_id: int, device: torch.device, model_type: str = "bilstm_cnn_pgn"):
    generator.eval()
    all_predicted_summaries = []
    all_reference_summaries = []

    pbar = tqdm(dataloader, desc="Generating summaries")
    for batch in pbar:
        src = batch['article'].to(device)
        trg_ref = batch['highlight'].to(device) # Keep reference for evaluation
        src_len = batch['article_len'].to(device)

        src_for_copy = src # Use source token IDs for PGN copy mechanism

        generated_ids = generator.generate(src, src_len, src_for_copy)

        predicted_texts = convert_ids_to_text(sp_tokenizer, generated_ids, eos_id, pad_id)
        reference_texts = convert_ids_to_text(sp_tokenizer, trg_ref, eos_id, pad_id)

        all_predicted_summaries.extend(predicted_texts)
        all_reference_summaries.extend(reference_texts)

    print("\nCalculating ROUGE scores...")
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': {'fmeasure': 0, 'precision': 0, 'recall': 0},
                    'rouge2': {'fmeasure': 0, 'precision': 0, 'recall': 0},
                    'rougeL': {'fmeasure': 0, 'precision': 0, 'recall': 0}}

    for i in tqdm(range(len(all_predicted_summaries)), desc="Computing ROUGE per sample"):
        scores = scorer.score(all_reference_summaries[i], all_predicted_summaries[i])
        for metric in rouge_scores:
            rouge_scores[metric]['fmeasure'] += scores[metric].fmeasure
            rouge_scores[metric]['precision'] += scores[metric].precision
            rouge_scores[metric]['recall'] += scores[metric].recall

    for metric in rouge_scores:
        for score_type in rouge_scores[metric]:
            rouge_scores[metric][score_type] /= len(all_predicted_summaries)

    print("\nROUGE Scores:")
    for metric, scores in rouge_scores.items():
        print(f"  {metric.upper()}:")
        print(f"    F1: {scores['fmeasure']:.4f}")
        print(f"    Precision: {scores['precision']:.4f}")
        print(f"    Recall: {scores['recall']:.4f}")

    print("\nCalculating BERTScore...")
    # BERTScore requires non-empty candidate and reference lists
    filtered_predicted_summaries = [s for s in all_predicted_summaries if s.strip()]
    filtered_reference_summaries = [s for s in all_reference_summaries if s.strip()] # BERTScore will error if reference is empty

    if not filtered_predicted_summaries or not filtered_reference_summaries:
        bert_score_results = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        print("Skipping BERTScore: No valid predicted or reference summaries to evaluate.")
    else:
        P, R, F1 = bert_score_calc(filtered_predicted_summaries, filtered_reference_summaries,
                                   lang="en", verbose=True, device=str(device))
        bert_score_results = {
            'precision': P.mean().item(),
            'recall': R.mean().item(),
            'f1': F1.mean().item()
        }
        print(f"\nBERTScore:")
        print(f"  Precision: {bert_score_results['precision']:.4f}")
        print(f"  Recall: {bert_score_results['recall']:.4f}")
        print(f"  F1: {bert_score_results['f1']:.4f}")

    print("\nCalculating METEOR score...")
    meteor_scores = []
    for i in tqdm(range(len(all_predicted_summaries)), desc="Computing METEOR per sample"):
        if not all_predicted_summaries[i].strip() or not all_reference_summaries[i].strip():
            meteor_scores.append(0.0)
            continue
        reference_tokenized = [nltk.word_tokenize(all_reference_summaries[i])]
        hypothesis_tokenized = nltk.word_tokenize(all_predicted_summaries[i])
        score = meteor_score(reference_tokenized, hypothesis_tokenized)
        meteor_scores.append(score)
    avg_meteor_score = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0
    print(f"\nMETEOR Score: {avg_meteor_score:.4f}")


    print("\nCalculating Readability and Cohesion scores...")
    readability_scores = {
        'flesch_reading_ease': [], 'flesch_kincaid_grade': [], 'dale_chall_readability_score': [],
        'automated_readability_index': [], 'coleman_liau_index': [], 'linsear_write_formula': [],
        'gunning_fog_score': [], 'smog_index': []
    }
    cohesion_scores = []

    for summary_text in tqdm(all_predicted_summaries, desc="Computing Readability & Cohesion"):
        if summary_text and len(summary_text.split()) > 0:
            try: # Use try-except for textstat functions as they can raise errors on malformed text
                readability_scores['flesch_reading_ease'].append(textstat.flesch_reading_ease(summary_text))
                readability_scores['flesch_kincaid_grade'].append(textstat.flesch_kincaid_grade(summary_text))
                readability_scores['dale_chall_readability_score'].append(textstat.dale_chall_readability_score(summary_text))
                readability_scores['automated_readability_index'].append(textstat.automated_readability_index(summary_text))
                readability_scores['coleman_liau_index'].append(textstat.coleman_liau_index(summary_text))
                readability_scores['linsear_write_formula'].append(textstat.linsear_write_formula(summary_text))
                readability_scores['gunning_fog_score'].append(textstat.gunning_fog(summary_text))
                readability_scores['smog_index'].append(textstat.smog_index(summary_text))
            except Exception as e:
                # print(f"Error calculating readability for text: '{summary_text[:50]}...' Error: {e}")
                for key in readability_scores: readability_scores[key].append(0.0) # Append default 0
        else:
            for key in readability_scores:
                readability_scores[key].append(0.0)

        cohesion_scores.append(calculate_cohesion_jaccard(summary_text))

    avg_readability_scores = {k: sum(v) / len(v) if v else 0.0 for k, v in readability_scores.items()}
    avg_cohesion_score = sum(cohesion_scores) / len(cohesion_scores) if cohesion_scores else 0.0

    print("\nReadability Scores (Average):")
    for metric, score in avg_readability_scores.items():
        print(f"  {metric.replace('_', ' ').title()}: {score:.4f}")
    print(f"\nCohesion Score (Jaccard Similarity Average): {avg_cohesion_score:.4f}")


    evaluation_summary = {
        'model_type': model_type,
        'rouge_scores': rouge_scores,
        'bert_score': bert_score_results,
        'meteor_score': avg_meteor_score,
        'readability_scores': avg_readability_scores,
        'cohesion_score_jaccard': avg_cohesion_score,
        'num_samples_evaluated': len(all_predicted_summaries)
    }

    output_filename = f"evaluation_scores_{model_type}.json"
    predicted_filename = f"predicted_summaries_{model_type}.txt"

    with open(f"{RESULTS_DIR}/{predicted_filename}", "w", encoding="utf-8") as f:
        for pred in all_predicted_summaries:
            f.write(pred + "\n")
    print(f"Predicted summaries saved to {RESULTS_DIR}/{predicted_filename}")

    with open(f"{RESULTS_DIR}/{output_filename}", "w", encoding="utf-8") as f:
        json.dump(evaluation_summary, f, indent=4)
    print(f"Evaluation scores saved to {RESULTS_DIR}/{output_filename}")


if __name__ == "__main__":
    test_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/test_processed.pt")
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False)

    # Load SentencePiece tokenizer
    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer_path = f"{BASE_DIR}/tokenizer.model"
    if not os.path.exists(sp_tokenizer_path):
        print(f"Error: SentencePiece tokenizer model not found at {sp_tokenizer_path}.")
        print("Please ensure your tokenizer is trained and saved.")
        exit()
    sp_tokenizer.load(sp_tokenizer_path)

    # Get essential IDs and vocab size from dataset
    PAD_ID = test_dataset.pad_id
    BOS_ID = test_dataset.bos_id
    EOS_ID = test_dataset.eos_id
    VOCAB_SIZE = test_dataset.vocab_size

    # Load simulated GloVe embeddings (same as in training)
    # Ensure this matches the GloVe loading logic in model/bilstm_cnn_pgn_model.py
    word_to_idx = {sp_tokenizer.id_to_piece(i): i for i in range(VOCAB_SIZE)}
    glove_embedding_matrix = load_glove_embeddings(VOCAB_SIZE, GLOVE_EMBEDDING_DIM, word_to_idx=word_to_idx)

    # Load the trained model checkpoint
    bilstm_cnn_pgn_model_path = f"{CHECKPOINTS_DIR}/bilstm_cnn_pgn_best_model.pt"
    
    if not os.path.exists(bilstm_cnn_pgn_model_path):
        print(f"Error: BiLSTM-CNN PGN model checkpoint not found at {bilstm_cnn_pgn_model_path}.")
        print("Please run train_bilstm_cnn_pgn.py first to train the model.")
        exit()

    print(f"Loading BiLSTM-CNN PGN model from {bilstm_cnn_pgn_model_path} for evaluation...")
    checkpoint = torch.load(bilstm_cnn_pgn_model_path, map_location=DEVICE)

    # Re-instantiate model components with correct parameters from checkpoint
    encoder = BiLSTMCNNEncoder(VOCAB_SIZE, GLOVE_EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID,
                               filter_sizes=FILTER_SIZES, num_filters=NUM_FILTERS,
                               embedding_matrix=glove_embedding_matrix).to(DEVICE)
    
    decoder = BiLSTMCNNPGNDecoder(VOCAB_SIZE, GLOVE_EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID,
                                  encoder_output_dim=HIDDEN_DIM * 2).to(DEVICE)
    
    model = Seq2SeqBiLSTMCNNPGN(encoder, decoder, DEVICE, PAD_ID).to(DEVICE)
    
    # Load state dictionaries into the model components
    model.encoder.load_state_dict(checkpoint['encoder_state_dict'])
    model.decoder.load_state_dict(checkpoint['decoder_state_dict'])
    
    print("BiLSTM-CNN PGN model loaded successfully.")

    # Create the generator for evaluation
    generator = BiLSTMCNNPGNGenerator(model, BOS_ID, EOS_ID, MAX_GENERATION_LENGTH, DEVICE).to(DEVICE)
    
    print("\nStarting evaluation of the BiLSTM-CNN PGN model...")
    evaluate_model_bilstm_cnn_pgn(generator, test_dataloader, sp_tokenizer, PAD_ID, EOS_ID, DEVICE, model_type="bilstm_cnn_pgn")
    print("\nEvaluation complete for the BiLSTM-CNN PGN model!")


Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/test_processed.pt for BiLSTM-CNN PGN evaluation...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Loaded 11490 samples for evaluation.
Simulating GloVe embedding loading. Creating a random embedding matrix of size (30000, 300).
Loading BiLSTM-CNN PGN model from /kaggle/working/TextSummarization/checkpoints_bilstm_cnn_pgn/bilstm_cnn_pgn_best_model.pt for evaluation...
BiLSTM-CNN PGN model loaded successfully.

Starting evaluation of the BiLSTM-CNN PGN model...


Generating summaries: 100%|██████████| 180/180 [01:09<00:00,  2.59it/s]



Calculating ROUGE scores...


...

done in 666.14 seconds, 17.25 sentences/sec

BERTScore:
  Precision: 0.8589
  Recall: 0.8378
  F1: 0.8481

Calculating METEOR score...


Computing METEOR per sample: 100%|██████████| 11490/11490 [00:07<00:00, 1593.35it/s]



METEOR Score: 0.0000

Calculating Readability and Cohesion scores...


Computing Readability & Cohesion: 100%|██████████| 11490/11490 [00:09<00:00, 1149.63it/s]


Readability Scores (Average):
  Flesch Reading Ease: -612.5911
  Flesch Kincaid Grade: 98.9519
  Dale Chall Readability Score: 19.4761
  Automated Readability Index: 805.6816
  Coleman Liau Index: 726.8087
  Linsear Write Formula: 0.3710
  Gunning Fog Score: 35.2407
  Smog Index: 8.1050

Cohesion Score (Jaccard Similarity Average): 0.0000
Predicted summaries saved to /kaggle/working/TextSummarization/results/predicted_summaries_bilstm_cnn_pgn.txt
Evaluation scores saved to /kaggle/working/TextSummarization/results/evaluation_scores_bilstm_cnn_pgn.json

Evaluation complete for the BiLSTM-CNN PGN model!


In [3]:
import os

base_dir = "/kaggle/working/TextSummarization"
model_dir = os.path.join(base_dir, "model")
dataset_dir = os.path.join(base_dir, "dataset")

os.makedirs(model_dir, exist_ok=True)
os.makedirs(dataset_dir, exist_ok=True)

# Create __init__.py files to make them Python packages
with open(os.path.join(model_dir, "__init__.py"), "w") as f:
    pass
with open(os.path.join(dataset_dir, "__init__.py"), "w") as f:
    pass

print(f"Directories created: {model_dir}, {dataset_dir}")
print("`__init__.py` files created in model and dataset directories.")


Directories created: /kaggle/working/TextSummarization/model, /kaggle/working/TextSummarization/dataset
`__init__.py` files created in model and dataset directories.


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    """
    The Encoder module of the BiLSTM-based text summarization model.
    It processes input articles, incorporating token, POS, DEP, and NER embeddings.
    """
    def __init__(self,
                 vocab_size: int,
                 pos_vocab_size: int,
                 dep_vocab_size: int,
                 ner_vocab_size: int,
                 embedding_dim: int,
                 hidden_dim: int,
                 n_layers: int,
                 dropout: float):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        # Linguistic feature embeddings
        feature_embedding_dim = embedding_dim // 3 # Roughly equally split
        self.pos_embedding = nn.Embedding(pos_vocab_size, feature_embedding_dim)
        self.dep_embedding = nn.Embedding(dep_vocab_size, feature_embedding_dim)
        self.ner_embedding = nn.Embedding(ner_vocab_size, feature_embedding_dim)

        # Total input dimension for the LSTM after concatenating all embeddings
        self.total_input_dim = embedding_dim + (3 * feature_embedding_dim)


        # BiLSTM layer
        self.rnn = nn.LSTM(self.total_input_dim,
                           hidden_dim,
                           num_layers=n_layers,
                           bidirectional=True,
                           dropout=dropout if n_layers > 1 else 0) # Apply dropout only if multiple layers

        self.dropout = nn.Dropout(dropout)

    def forward(self,
                src_tokens: torch.Tensor,
                src_pos: torch.Tensor,
                src_dep: torch.Tensor,
                src_ner: torch.Tensor):
        """
        Forward pass for the encoder.

        Args:
            src_tokens (torch.Tensor): Input token IDs of shape [seq_len, batch_size].
            src_pos (torch.Tensor): POS tag IDs of shape [seq_len, batch_size].
            src_dep (torch.Tensor): Dependency tag IDs of shape [seq_len, batch_size].
            src_ner (torch.Tensor): NER tag IDs of shape [seq_len, batch_size].

        Returns:
            tuple: A tuple containing:
                - outputs (torch.Tensor): Encoder outputs for each timestep [seq_len, batch_size, hidden_dim * 2].
                - hidden (torch.Tensor): Final hidden state [n_layers * 2, batch_size, hidden_dim].
                - cell (torch.Tensor): Final cell state [n_layers * 2, batch_size, hidden_dim].
        """
        # Embed all inputs
        token_embedded = self.dropout(self.token_embedding(src_tokens))
        pos_embedded = self.dropout(self.pos_embedding(src_pos))
        dep_embedded = self.dropout(self.dep_embedding(src_dep))
        ner_embedded = self.dropout(self.ner_embedding(src_ner))

        # Concatenate embeddings along the last dimension (feature dimension)
        combined_embedded = torch.cat((token_embedded, pos_embedded, dep_embedded, ner_embedded), dim=2)

        # outputs: [seq_len, batch_size, hidden_dim * num_directions]
        # hidden, cell: [num_layers * num_directions, batch_size, hidden_dim]
        outputs, (hidden, cell) = self.rnn(combined_embedded)

        return outputs, hidden, cell


class Attention(nn.Module):
    """
    Luong-style multiplicative attention mechanism.
    Calculates attention scores and applies them to encoder outputs.
    """
    def __init__(self, enc_hidden_dim: int, dec_hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(enc_hidden_dim + dec_hidden_dim, dec_hidden_dim)
        self.v = nn.Linear(dec_hidden_dim, 1, bias=False) # Context vector for scoring

    def forward(self, hidden: torch.Tensor, encoder_outputs: torch.Tensor):
        """
        Forward pass for attention.

        Args:
            hidden (torch.Tensor): Current decoder hidden state [1, batch_size, dec_hidden_dim].
                                   (We'll use the last layer's hidden state for attention)
            encoder_outputs (torch.Tensor): All encoder outputs [src_len, batch_size, enc_hidden_dim * 2].

        Returns:
            torch.Tensor: Attention weights [batch_size, src_len].
        """
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]

        # repeat decoder hidden state src_len times
        # hidden = [1, batch_size, dec_hidden_dim] -> [src_len, batch_size, dec_hidden_dim]
        hidden = hidden.repeat(src_len, 1, 1)

        # Concatenate hidden state with encoder outputs
        # energy = [src_len, batch_size, enc_hidden_dim * 2 + dec_hidden_dim]
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))

        # Apply context vector to get scores
        # attention = [src_len, batch_size, 1]
        attention = self.v(energy).squeeze(2)

        # Transpose to [batch_size, src_len] and apply softmax
        return F.softmax(attention.permute(1, 0), dim=1)


class Decoder(nn.Module):
    """
    The Decoder module of the BiLSTM-based text summarization model.
    It generates output tokens one by one, using an attention mechanism.
    """
    def __init__(self,
                 vocab_size: int,
                 embedding_dim: int,
                 hidden_dim: int, # This is DECODER_HIDDEN_DIM, which is ENCODER_HIDDEN_DIM * 2
                 n_layers: int,
                 dropout: float,
                 attention: nn.Module):
        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.attention = attention

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Corrected input size for the decoder's LSTM
        # It's embedding_dim + the dimension of the context vector (which is hidden_dim from encoder * 2)
        # Since hidden_dim for decoder is already hidden_dim_encoder * 2, this simplifies.
        self.rnn = nn.LSTM(embedding_dim + hidden_dim, # Corrected: hidden_dim is already encoder_hidden_dim * 2
                           hidden_dim,
                           num_layers=n_layers,
                           dropout=dropout if n_layers > 1 else 0)

        # Linear layer to output vocabulary probabilities
        self.fc_out = nn.Linear(embedding_dim + hidden_dim + hidden_dim, vocab_size) # embedding + output of decoder RNN + context

        self.dropout = nn.Dropout(dropout)

    def forward(self,
                trg: torch.Tensor,
                encoder_outputs: torch.Tensor,
                hidden: torch.Tensor,
                cell: torch.Tensor):
        """
        Forward pass for the decoder.

        Args:
            trg (torch.Tensor): Current target token (usually single token) [1, batch_size].
            encoder_outputs (torch.Tensor): All encoder outputs [src_len, batch_size, hidden_dim * 2].
            hidden (torch.Tensor): Previous decoder hidden state [n_layers, batch_size, hidden_dim].
            cell (torch.Tensor): Previous decoder cell state [n_layers, batch_size, hidden_dim].

        Returns:
            tuple: A tuple containing:
                - output (torch.Tensor): Predicted next token probabilities [batch_size, vocab_size].
                - hidden (torch.Tensor): Current hidden state [n_layers, batch_size, hidden_dim].
                - cell (torch.Tensor): Current cell state [n_layers, batch_size, hidden_dim].
        """
        # trg = [1, batch_size]
        embedded = self.dropout(self.embedding(trg)) # embedded = [1, batch_size, embedding_dim]

        # Use the hidden state from the last layer for attention
        # For multi-layer LSTM, hidden state is [num_layers, batch_size, hidden_dim]
        # We need the hidden state of the *last* layer, which is hidden[-1, :, :]
        # But attention expects [1, batch_size, hidden_dim]
        # So we take the last layer and unsqueeze
        attn_hidden = hidden[-1, :, :].unsqueeze(0) # [1, batch_size, hidden_dim]

        # attention_weights = [batch_size, src_len]
        attention_weights = self.attention(attn_hidden, encoder_outputs)

        # attention_weights = [batch_size, 1, src_len]
        attention_weights = attention_weights.unsqueeze(1)

        # encoder_outputs = [src_len, batch_size, enc_hidden_dim * 2]
        # context = [batch_size, 1, enc_hidden_dim * 2]
        context = torch.bmm(attention_weights, encoder_outputs.permute(1, 0, 2))

        # context = [1, batch_size, enc_hidden_dim * 2]
        context = context.permute(1, 0, 2)

        # Concatenate embedded input and context vector
        # rnn_input = [1, batch_size, embedding_dim + (encoder_hidden_dim * 2)]
        rnn_input = torch.cat((embedded, context), dim=2)

        # output: [1, batch_size, hidden_dim]
        # hidden, cell: [n_layers, batch_size, hidden_dim]
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        # Pass through linear layer to get vocabulary probabilities
        # prediction = [batch_size, vocab_size]
        # It concatenates:
        # 1. output.squeeze(0)          -> [batch_size, hidden_dim] (from decoder RNN)
        # 2. embedded.squeeze(0)        -> [batch_size, embedding_dim] (current token embedding)
        # 3. context.squeeze(0)         -> [batch_size, encoder_hidden_dim * 2] (context vector)
        prediction = self.fc_out(torch.cat((output.squeeze(0),
                                              embedded.squeeze(0),
                                              context.squeeze(0)), dim=1))

        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    """
    The complete Sequence-to-Sequence model combining the Encoder and Decoder.
    It takes an input article and generates a summary.
    """
    def __init__(self, encoder: nn.Module, decoder: nn.Module, device: torch.device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        # Ensure encoder's combined hidden dimension (from bidirectional) matches decoder's hidden dimension
        assert encoder.hidden_dim * 2 == decoder.hidden_dim, \
            "Decoder hidden dimension must be double the encoder's hidden dimension for direct transfer from bidirectional encoder!"
        # We are no longer asserting on the number of layers directly here, as the initialization logic
        # now handles repeating the combined last encoder states across decoder layers.


    def forward(self,
                src_tokens: torch.Tensor,
                src_pos: torch.Tensor,
                src_dep: torch.Tensor,
                src_ner: torch.Tensor,
                trg_tokens: torch.Tensor,
                teacher_forcing_ratio: float = 0.5):
        """
        Forward pass for the Seq2Seq model.

        Args:
            src_tokens (torch.Tensor): Source token IDs [src_len, batch_size].
            src_pos (torch.Tensor): Source POS tag IDs [src_len, batch_size].
            src_dep (torch.Tensor): Source Dependency tag IDs [src_len, batch_size].
            src_ner (torch.Tensor): Source NER tag IDs [src_len, batch_size].
            trg_tokens (torch.Tensor): Target token IDs [trg_len, batch_size].
            teacher_forcing_ratio (float): Probability to use actual target output as next input.

        Returns:
            torch.Tensor: Predicted output token probabilities for each timestep [trg_len, batch_size, vocab_size].
        """
        batch_size = src_tokens.shape[1]
        trg_len = trg_tokens.shape[0]
        vocab_size = self.decoder.vocab_size

        # Tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, vocab_size).to(self.device)

        # encoder_outputs: [src_len, batch_size, hidden_dim * 2] (from bidirectional)
        # hidden, cell: [n_layers * 2, batch_size, hidden_dim]
        encoder_outputs, hidden, cell = self.encoder(src_tokens, src_pos, src_dep, src_ner)

        # Initialize decoder's hidden and cell states
        # Take the final hidden/cell state from the *last* layer of both forward and backward LSTMs.
        # hidden[-2, :, :] is the last forward layer's hidden state.
        # hidden[-1, :, :] is the last backward layer's hidden state.
        # Concatenate them to form a single initial state for the decoder,
        # which will have dimensions [batch_size, hidden_dim * 2].
        # Then unsqueeze to add the layer dimension [1, batch_size, hidden_dim * 2].
        initial_hidden_state_decoder = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1).unsqueeze(0)
        initial_cell_state_decoder = torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1).unsqueeze(0)

        # Repeat this initial state for all decoder layers.
        # The decoder's hidden and cell states are [n_layers, batch_size, hidden_dim].
        hidden_decoder = initial_hidden_state_decoder.repeat(self.decoder.n_layers, 1, 1)
        cell_decoder = initial_cell_state_decoder.repeat(self.decoder.n_layers, 1, 1)

        # First input to the decoder is the <bos> token
        input_token = trg_tokens[0, :] # [batch_size]

        # Loop through the target sequence length
        for t in range(1, trg_len):
            # Pass input_token (unsqueeze to add sequence length dim), encoder_outputs,
            # and current hidden/cell states to decoder
            # output: [batch_size, vocab_size]
            # hidden_decoder, cell_decoder: [n_layers, batch_size, hidden_dim]
            output, hidden_decoder, cell_decoder = self.decoder(input_token.unsqueeze(0),
                                                                 encoder_outputs,
                                                                 hidden_decoder,
                                                                 cell_decoder)

            # Store the prediction
            outputs[t] = output

            # Decide whether to use teacher forcing or not
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1) # Get the highest predicted token ID

            # If teacher forcing, use actual next token. Else, use predicted token.
            input_token = trg_tokens[t] if teacher_force else top1

        return outputs

In [8]:
import torch
from torch.utils.data import Dataset
import os

class SummarizationDataset(Dataset):
    """
    Custom PyTorch Dataset for loading preprocessed text summarization data.
    It loads token IDs (articles, highlights) and linguistic feature IDs (POS, DEP, NER).
    """
    def __init__(self, data_path: str):
        """
        Initializes the dataset by loading the preprocessed .pt file.

        Args:
            data_path (str): Path to the .pt file containing processed data.
        """
        if not os.path.exists(data_path):
            raise FileNotFoundError(f"Data file not found at {data_path}")

        print(f"Loading data from {data_path}...")
        self.data = torch.load(data_path)
        print("Data loaded successfully.")

        self.articles_ids = self.data['articles_ids']
        self.highlights_ids = self.data['highlights_ids']
        self.articles_pos_ids = self.data['articles_pos_ids']
        self.highlights_pos_ids = self.data['highlights_pos_ids']
        self.articles_dep_ids = self.data['articles_dep_ids']
        self.highlights_dep_ids = self.data['highlights_dep_ids']
        self.articles_ner_ids = self.data['articles_ner_tags_word_level'] # Corrected key
        self.highlights_ner_ids = self.data['highlights_ner_tags_word_level'] # Corrected key

        self.pad_id = self.data['pad_id']
        self.bos_id = self.data['bos_id']
        self.eos_id = self.data['eos_id']
        self.vocab_size = self.data['vocab_size']
        self.pos_vocab_size = self.data['pos_vocab_size']
        self.dep_vocab_size = self.data['dep_vocab_size']
        self.ner_vocab_size = self.data['ner_vocab_size']

        # Ensure all tensors have the same number of samples
        num_samples = len(self.articles_ids)
        assert len(self.highlights_ids) == num_samples
        assert len(self.articles_pos_ids) == num_samples
        assert len(self.highlights_pos_ids) == num_samples
        assert len(self.articles_dep_ids) == num_samples
        assert len(self.highlights_dep_ids) == num_samples
        assert len(self.articles_ner_ids) == num_samples
        assert len(self.highlights_ner_ids) == num_samples


    def __len__(self) -> int:
        """
        Returns the total number of samples in the dataset.
        """
        return len(self.articles_ids)

    def __getitem__(self, idx: int) -> dict:
        """
        Retrieves a single sample from the dataset at the given index.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            dict: A dictionary containing:
                'article_ids': Token IDs for the article.
                'highlight_ids': Token IDs for the highlight (summary).
                'article_pos_ids': POS tag IDs for the article.
                'highlight_pos_ids': POS tag IDs for the highlight.
                'article_dep_ids': Dependency tag IDs for the article.
                'highlight_dep_ids': Dependency tag IDs for the highlight.
                'article_ner_ids': NER tag IDs for the article.
                'highlight_ner_ids': NER tag IDs for the highlight.
        """
        return {
            'article_ids': self.articles_ids[idx],
            'highlight_ids': self.highlights_ids[idx],
            'article_pos_ids': self.articles_pos_ids[idx],
            'highlight_pos_ids': self.highlights_pos_ids[idx],
            'article_dep_ids': self.articles_dep_ids[idx],
            'highlight_dep_ids': self.highlights_dep_ids[idx],
            'article_ner_ids': self.articles_ner_ids[idx],
            'highlight_ner_ids': self.highlights_ner_ids[idx]
        }



In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import math
import time

# Define constants and paths
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
RESULTS_DIR = f"{BASE_DIR}/results"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model" # To load PAD_ID etc.

# Hyperparameters (you might need to tune these)
EMBEDDING_DIM = 256
ENCODER_HIDDEN_DIM = 512 # Hidden dimension for a single direction of Encoder's LSTM
DECODER_HIDDEN_DIM = ENCODER_HIDDEN_DIM * 2 # Decoder's hidden dim must be double encoder's for direct transfer
ENCODER_LAYERS = 2
DECODER_LAYERS = 4 # Should be equal to ENCODER_LAYERS * 2 for direct transfer of states
ENCODER_DROPOUT = 0.5
DECODER_DROPOUT = 0.5
BATCH_SIZE = 32
N_EPOCHS = 10
LEARNING_RATE = 0.001
CLIP = 1.0 # Gradient clipping

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def train_epoch(model: nn.Module,
                data_loader: DataLoader,
                optimizer: optim.Optimizer,
                criterion: nn.Module,
                clip: float,
                pad_idx: int):
    """
    Performs one epoch of training.

    Args:
        model (nn.Module): The Seq2Seq model.
        data_loader (DataLoader): DataLoader for training data.
        optimizer (optim.Optimizer): Optimizer.
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss).
        clip (float): Gradient clipping value.
        pad_idx (int): Padding token ID.

    Returns:
        float: Average loss for the epoch.
    """
    model.train() # Set model to training mode
    epoch_loss = 0

    for batch in tqdm(data_loader, desc="Training"):
        src_tokens = batch['article_ids'].permute(1, 0).to(device) # [seq_len, batch_size]
        trg_tokens = batch['highlight_ids'].permute(1, 0).to(device) # [seq_len, batch_size]
        src_pos = batch['article_pos_ids'].permute(1, 0).to(device)
        src_dep = batch['article_dep_ids'].permute(1, 0).to(device)
        src_ner = batch['article_ner_ids'].permute(1, 0).to(device)

        optimizer.zero_grad() # Clear gradients

        # Output from model: [trg_len, batch_size, vocab_size]
        output = model(src_tokens, src_pos, src_dep, src_ner, trg_tokens)

        # trg_len is target sequence length, vocab_size is size of vocabulary
        # For loss calculation, we need to flatten the output and target
        # Skip the first token in target (BOS token) as we're predicting the next tokens
        # Output: [(trg_len - 1) * batch_size, vocab_size]
        # Target: [(trg_len - 1) * batch_size]
        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim) # Changed .view to .reshape
        trg_tokens = trg_tokens[1:].reshape(-1)     # Changed .view to .reshape

        loss = criterion(output, trg_tokens) # Calculate loss

        loss.backward() # Backpropagation

        # Clip gradients to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step() # Update model parameters

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)


def evaluate_epoch(model: nn.Module,
                   data_loader: DataLoader,
                   criterion: nn.Module,
                   pad_idx: int):
    """
    Evaluates the model on the validation/test set.

    Args:
        model (nn.Module): The Seq2Seq model.
        data_loader (DataLoader): DataLoader for evaluation data.
        criterion (nn.Module): Loss function.
        pad_idx (int): Padding token ID.

    Returns:
        float: Average loss for the epoch.
    """
    model.eval() # Set model to evaluation mode
    epoch_loss = 0

    with torch.no_grad(): # Disable gradient calculations
        for batch in tqdm(data_loader, desc="Evaluating"):
            src_tokens = batch['article_ids'].permute(1, 0).to(device)
            trg_tokens = batch['highlight_ids'].permute(1, 0).to(device)
            src_pos = batch['article_pos_ids'].permute(1, 0).to(device)
            src_dep = batch['article_dep_ids'].permute(1, 0).to(device)
            src_ner = batch['article_ner_ids'].permute(1, 0).to(device)

            # Turn off teacher forcing during evaluation
            output = model(src_tokens, src_pos, src_dep, src_ner, trg_tokens, 0)

            output_dim = output.shape[-1]
            output = output[1:].reshape(-1, output_dim) # Changed .view to .reshape
            trg_tokens = trg_tokens[1:].reshape(-1)     # Changed .view to .reshape

            loss = criterion(output, trg_tokens)
            epoch_loss += loss.item()

    return epoch_loss / len(data_loader)


def main():
    # Load data
    train_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/train_processed.pt")
    val_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/validation_processed.pt")
    test_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/test_processed.pt")

    # Get vocabulary sizes from dataset
    VOCAB_SIZE = train_data.vocab_size
    POS_VOCAB_SIZE = train_data.pos_vocab_size
    DEP_VOCAB_SIZE = train_data.dep_vocab_size
    NER_VOCAB_SIZE = train_data.ner_vocab_size
    PAD_IDX = train_data.pad_id

    print(f"Vocab sizes: Token={VOCAB_SIZE}, POS={POS_VOCAB_SIZE}, DEP={DEP_VOCAB_SIZE}, NER={NER_VOCAB_SIZE}")
    print(f"Padding ID: {PAD_IDX}")

    # Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False) # For final evaluation

    # Initialize Encoder, Attention, Decoder, and Seq2Seq Model
    encoder = Encoder(VOCAB_SIZE,
                      POS_VOCAB_SIZE,
                      DEP_VOCAB_SIZE,
                      NER_VOCAB_SIZE,
                      EMBEDDING_DIM,
                      ENCODER_HIDDEN_DIM, # Use ENCODER_HIDDEN_DIM here
                      ENCODER_LAYERS,
                      ENCODER_DROPOUT)

    # Attention module needs encoder's output hidden dim (which is ENCODER_HIDDEN_DIM * 2)
    # and decoder's hidden dim (which is DECODER_HIDDEN_DIM)
    attention = Attention(ENCODER_HIDDEN_DIM * 2, DECODER_HIDDEN_DIM)

    decoder = Decoder(VOCAB_SIZE,
                      EMBEDDING_DIM,
                      DECODER_HIDDEN_DIM, # Use DECODER_HIDDEN_DIM here
                      DECODER_LAYERS,
                      DECODER_DROPOUT,
                      attention)

    model = Seq2Seq(encoder, decoder, device).to(device)

    # Initialize model weights (optional but good practice)
    def init_weights(m):
        for name, param in m.named_parameters():
            if 'weight' in name:
                nn.init.normal_(param.data, mean=0, std=0.01)
            else:
                nn.init.constant_(param.data, 0)
    model.apply(init_weights)

    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")

    # Optimizer and Loss Function
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    # CrossEntropyLoss ignores specified index when calculating loss, useful for padding
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    best_valid_loss = float('inf')

    print("\nStarting training...")
    for epoch in range(N_EPOCHS):
        start_time = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP, PAD_IDX)
        valid_loss = evaluate_epoch(model, val_loader, criterion, PAD_IDX)

        end_time = time.time()
        epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pt'))

        print(f'Epoch: {epoch+1:02} | Time: {int(epoch_mins)}m {int(epoch_secs)}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):.2f}')
        print(f'\tVal. Loss: {valid_loss:.3f} | Val. PPL: {math.exp(valid_loss):.2f}')

    # Load best model for final evaluation
    model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, 'best_model.pt')))
    test_loss = evaluate_epoch(model, test_loader, criterion, PAD_IDX)
    print(f'\nTest Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):.2f}')

    # --- ROUGE Score Calculation (Placeholder) ---
    # To calculate ROUGE scores, you'd typically implement a generation function
    # that uses the trained model to generate summaries for the test set,
    # then compare them to the reference highlights.
    # You would need to decode the generated token IDs back to text using your SentencePiece tokenizer.
    # Install the 'rouge_score' library: `pip install rouge_score`
    # from rouge_score import rouge_scorer
    # from your_tokenizer_module import load_tokenizer # You have this already

    # For example:
    # scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    # reference_summaries = ["This is a reference summary."]
    # generated_summaries = ["This is a generated summary."]
    # scores = scorer.score(reference_summaries[0], generated_summaries[0])
    # print(scores)

    print("\nTraining complete. Model saved to results directory.")

if __name__ == "__main__":
    main()

Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/train_processed.pt...
Data loaded successfully.
Loading data from /kaggle/working/TextSummarization/data/processed/validation_processed.pt...
Data loaded successfully.
Loading data from /kaggle/working/TextSummarization/data/processed/test_processed.pt...
Data loaded successfully.
Vocab sizes: Token=30000, POS=50, DEP=45, NER=19
Padding ID: 0
Model has 131,752,714 trainable parameters

Starting training...


Evaluating: 100%|██████████| 418/418 [04:41<00:00,  1.49it/s]


Epoch: 01 | Time: 25m 47s
	Train Loss: 7.553 | Train PPL: 1905.87
	Val. Loss: 7.473 | Val. PPL: 1760.49


Evaluating: 100%|██████████| 418/418 [04:43<00:00,  1.47it/s]


Epoch: 02 | Time: 25m 55s
	Train Loss: 7.088 | Train PPL: 1197.27
	Val. Loss: 7.419 | Val. PPL: 1668.12


...

Epoch: 09 | Time: 25m 55s
	Train Loss: 5.010 | Train PPL: 149.85
	Val. Loss: 7.849 | Val. PPL: 2563.43


Evaluating: 100%|██████████| 418/418 [04:41<00:00,  1.48it/s]


Epoch: 10 | Time: 25m 55s
	Train Loss: 4.802 | Train PPL: 121.74
	Val. Loss: 7.949 | Val. PPL: 2832.63


Evaluating: 100%|██████████| 360/360 [04:01<00:00,  1.49it/s]


Test Loss: 7.314 | Test PPL: 1500.48

Training complete. Model saved to results directory.


In [3]:
!pip install evaluate contractions textstat rouge_score bert_score transformers seqeval --
# --- Start of dependency installation/resolution ---
# Downgrade numpy to a version compatible with evaluate/transformers dependencies.
# This must run BEFORE any imports that depend on numpy (like torch, evaluate, etc.)
# Using a specific 1.x version that is known to work with common libraries.
import os
print("Ensuring NumPy compatibility...")
os.system("pip install numpy==1.26.4 --force-reinstall")

# Install bert-score, which is a dependency for evaluate.load('bertscore')
print("Installing bert-score...")
os.system("pip install bert-score")

Ensuring NumPy compatibility...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 79.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.40.1 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but y

Installing bert-score...


0

In [13]:


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import math

# For evaluation metrics
import evaluate
from rouge_score import rouge_scorer
import nltk
import sentencepiece as spm
import textstat # Added textstat for readability

# Download NLTK data if not already present for METEOR and BLEU
try:
    nltk.data.find('corpora/wordnet')
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/omw-1.4')
except LookupError: # Changed exception type from nltk.downloader.DownloadError to LookupError
    print("Downloading NLTK data (wordnet, punkt, omw-1.4)... This may take a moment.")
    nltk.download('wordnet')
    nltk.download('punkt')
    nltk.download('omw-1.4')
    print("NLTK data downloaded.")
except Exception as e:
    print(f"An unexpected error occurred during NLTK data check/download: {e}")
# --- End of dependency installation/resolution ---

# Define constants and paths (must be consistent with training)
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
RESULTS_DIR = f"{BASE_DIR}/results"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model"
MODEL_PATH = os.path.join(RESULTS_DIR, 'best_model.pt') # Path to your trained model

# Hyperparameters (must match the model you trained)
EMBEDDING_DIM = 256
ENCODER_HIDDEN_DIM = 512
DECODER_HIDDEN_DIM = ENCODER_HIDDEN_DIM * 2
ENCODER_LAYERS = 2
DECODER_LAYERS = 4
ENCODER_DROPOUT = 0.5
DECODER_DROPOUT = 0.5
BATCH_SIZE = 64 # Can be adjusted for evaluation, but keep it consistent for stability
MAX_LEN = 64 # Max sequence length, needs to be consistent with preprocessing

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def generate_summary(model: nn.Module,
                     src_tokens: torch.Tensor,
                     src_pos: torch.Tensor,
                     src_dep: torch.Tensor,
                     src_ner: torch.Tensor,
                     sp_tokenizer: spm.SentencePieceProcessor,
                     max_len: int,
                     bos_id: int,
                     eos_id: int,
                     device: torch.device):
    """
    Generates a summary for a single input article using greedy decoding.

    Args:
        model (nn.Module): The Seq2Seq model.
        src_tokens (torch.Tensor): Input article token IDs [seq_len, 1] (single sample).
        src_pos (torch.Tensor): Input article POS IDs [seq_len, 1].
        src_dep (torch.Tensor): Input article DEP IDs [seq_len, 1].
        src_ner (torch.Tensor): Input article NER IDs [seq_len, 1].
        sp_tokenizer: SentencePiece tokenizer for decoding.
        max_len (int): Maximum length of the generated summary.
        bos_id (int): Beginning-of-sentence token ID.
        eos_id (int): End-of-sentence token ID.
        device (torch.device): Device to perform computations on.

    Returns:
        str: The generated summary text.
    """
    model.eval()
    with torch.no_grad():
        # Encode the source sequence
        encoder_outputs, hidden, cell = model.encoder(src_tokens, src_pos, src_dep, src_ner)

        # Initialize decoder's hidden and cell states
        initial_hidden_state_decoder = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1).unsqueeze(0)
        initial_cell_state_decoder = torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1).unsqueeze(0)

        hidden_decoder = initial_hidden_state_decoder.repeat(model.decoder.n_layers, 1, 1)
        cell_decoder = initial_cell_state_decoder.repeat(model.decoder.n_layers, 1, 1)

        # Start with BOS token
        input_token = torch.tensor([[bos_id]], device=device) # [1, 1]

        generated_tokens = []
        for _ in range(max_len):
            output, hidden_decoder, cell_decoder = model.decoder(input_token,
                                                                 encoder_outputs,
                                                                 hidden_decoder,
                                                                 cell_decoder)
            # Get the token with the highest probability
            predicted_token = output.argmax(1).item()
            generated_tokens.append(predicted_token)

            # If EOS is predicted, stop generating
            if predicted_token == eos_id:
                break

            # Use the predicted token as the input for the next step
            input_token = torch.tensor([[predicted_token]], device=device)

    # Decode the generated token IDs
    # Remove BOS if present at the start (shouldn't be, as we start with BOS for prediction)
    # Remove EOS if present at the end
    if generated_tokens and generated_tokens[0] == bos_id:
        generated_tokens = generated_tokens[1:]
    if generated_tokens and generated_tokens[-1] == eos_id:
        generated_tokens = generated_tokens[:-1]

    # Convert token IDs to text
    generated_text = sp_tokenizer.decode_ids(generated_tokens)
    return generated_text


def calculate_metrics(references: list[str], predictions: list[str]):
    """
    Calculates ROUGE, BLEU, METEOR, BERTScore, and Readability metrics.

    Args:
        references (list[str]): List of reference summaries.
        predictions (list[str]): List of generated summaries.

    Returns:
        dict: A dictionary containing the calculated scores.
    """
    metrics = {}

    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    # Initialize all ROUGE scores with 0.0
    rouge_total_scores = {'rouge1': {'fmeasure': 0.0, 'precision': 0.0, 'recall': 0.0},
                          'rouge2': {'fmeasure': 0.0, 'precision': 0.0, 'recall': 0.0},
                          'rougeL': {'fmeasure': 0.0, 'precision': 0.0, 'recall': 0.0}}

    for ref, pred in zip(references, predictions):
        score = scorer.score(ref, pred)
        for k in rouge_total_scores:
            rouge_total_scores[k]['fmeasure'] += score[k].fmeasure
            rouge_total_scores[k]['precision'] += score[k].precision
            rouge_total_scores[k]['recall'] += score[k].recall
    
    for k in rouge_total_scores:
        metrics[k + '_f1'] = rouge_total_scores[k]['fmeasure'] / len(references)
        metrics[k + '_precision'] = rouge_total_scores[k]['precision'] / len(references)
        metrics[k + '_recall'] = rouge_total_scores[k]['recall'] / len(references)


    # BLEU
    bleu_metric = evaluate.load("bleu")
    bleu_results = bleu_metric.compute(predictions=predictions, references=references)
    metrics['bleu'] = bleu_results['bleu']

    # METEOR
    meteor_metric = evaluate.load("meteor")
    meteor_results = meteor_metric.compute(predictions=predictions, references=references)
    metrics['meteor'] = meteor_results['meteor']

    # BERTScore
    bertscore_metric = evaluate.load("bertscore")
    # Using 'distilbert-base-uncased' for faster computation.
    # For higher quality, consider 'roberta-large' but it requires more memory.
    bertscore_results = bertscore_metric.compute(
        predictions=predictions, references=references, lang="en", model_type="distilbert-base-uncased"
    )
    # Get the average F1, precision, recall scores
    metrics['bertscore_f1'] = sum(bertscore_results['f1']) / len(bertscore_results['f1'])
    metrics['bertscore_precision'] = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
    metrics['bertscore_recall'] = sum(bertscore_results['recall']) / len(bertscore_results['recall'])


    # Readability Scores (averaged over all generated summaries)
    flesch_reading_ease_scores = []
    flesch_kincaid_grade_scores = []
    dale_chall_readability_scores = []
    automated_readability_index_scores = []
    coleman_liau_index_scores = []
    linsear_write_formula_scores = []
    gunning_fog_scores = []
    smog_index_scores = []

    for pred in predictions:
        # Handle cases where textstat might fail on empty or very short strings
        if not pred.strip():
            continue # Skip empty summaries

        try:
            flesch_reading_ease_scores.append(textstat.flesch_reading_ease(pred))
            flesch_kincaid_grade_scores.append(textstat.flesch_kincaid_grade(pred))
            dale_chall_readability_scores.append(textstat.dale_chall_readability_score(pred))
            automated_readability_index_scores.append(textstat.automated_readability_index(pred))
            coleman_liau_index_scores.append(textstat.coleman_liau_index(pred))
            linsear_write_formula_scores.append(textstat.linsear_write_formula(pred))
            gunning_fog_scores.append(textstat.gunning_fog(pred))
            smog_index_scores.append(textstat.smog_index(pred))
        except Exception as e:
            # Catch any error during textstat calculation for a specific summary
            # print(f"Warning: Could not calculate readability for a summary. Error: {e}")
            pass # Continue processing other summaries

    # Calculate averages, handle empty lists if all summaries failed/were empty
    metrics['flesch_reading_ease'] = sum(flesch_reading_ease_scores) / len(flesch_reading_ease_scores) if flesch_reading_ease_scores else 0.0
    metrics['flesch_kincaid_grade'] = sum(flesch_kincaid_grade_scores) / len(flesch_kincaid_grade_scores) if flesch_kincaid_grade_scores else 0.0
    metrics['dale_chall_readability_score'] = sum(dale_chall_readability_scores) / len(dale_chall_readability_scores) if dale_chall_readability_scores else 0.0
    metrics['automated_readability_index'] = sum(automated_readability_index_scores) / len(automated_readability_index_scores) if automated_readability_index_scores else 0.0
    metrics['coleman_liau_index'] = sum(coleman_liau_index_scores) / len(coleman_liau_index_scores) if coleman_liau_index_scores else 0.0
    metrics['linsear_write_formula'] = sum(linsear_write_formula_scores) / len(linsear_write_formula_scores) if linsear_write_formula_scores else 0.0
    metrics['gunning_fog_score'] = sum(gunning_fog_scores) / len(gunning_fog_scores) if gunning_fog_scores else 0.0
    metrics['smog_index'] = sum(smog_index_scores) / len(smog_index_scores) if smog_index_scores else 0.0


    return metrics


def evaluate_model():
    """
    Loads the best trained model, evaluates it on the test set,
    and prints summarization metrics.
    """
    # Load test data
    test_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/test_processed.pt")
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    # Load SentencePiece tokenizer
    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer.load(TOKENIZER_MODEL_PATH)

    # Get vocabulary sizes and special token IDs from tokenizer
    VOCAB_SIZE = sp_tokenizer.get_piece_size()
    POS_VOCAB_SIZE = test_data.pos_vocab_size
    DEP_VOCAB_SIZE = test_data.dep_vocab_size
    NER_VOCAB_SIZE = test_data.ner_vocab_size
    PAD_IDX = sp_tokenizer.pad_id()
    BOS_IDX = sp_tokenizer.bos_id()
    EOS_IDX = sp_tokenizer.eos_id()

    print(f"Vocab sizes: Token={VOCAB_SIZE}, POS={POS_VOCAB_SIZE}, DEP={DEP_VOCAB_SIZE}, NER={NER_VOCAB_SIZE}")
    print(f"Special IDs: PAD={PAD_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")

    # Initialize Encoder, Attention, Decoder, and Seq2Seq Model
    encoder = Encoder(VOCAB_SIZE,
                      POS_VOCAB_SIZE,
                      DEP_VOCAB_SIZE,
                      NER_VOCAB_SIZE,
                      EMBEDDING_DIM,
                      ENCODER_HIDDEN_DIM,
                      ENCODER_LAYERS,
                      ENCODER_DROPOUT)

    attention = Attention(ENCODER_HIDDEN_DIM * 2, DECODER_HIDDEN_DIM)

    decoder = Decoder(VOCAB_SIZE,
                      EMBEDDING_DIM,
                      DECODER_HIDDEN_DIM,
                      DECODER_LAYERS,
                      DECODER_DROPOUT,
                      attention)

    model = Seq2Seq(encoder, decoder, device).to(device)

    # Load the best trained model weights
    if os.path.exists(MODEL_PATH):
        print(f"Loading model from {MODEL_PATH}")
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    else:
        print(f"Error: Model not found at {MODEL_PATH}. Please ensure you have trained the model first.")
        return

    # Set model to evaluation mode
    model.eval()

    # Prepare for metric calculation
    generated_summaries = []
    reference_summaries = []

    print("\nStarting evaluation on the test set...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating summaries"):
            src_tokens_batch = batch['article_ids'].permute(1, 0).to(device)
            trg_tokens_batch = batch['highlight_ids'].permute(1, 0).to(device)
            src_pos_batch = batch['article_pos_ids'].permute(1, 0).to(device)
            src_dep_batch = batch['article_dep_ids'].permute(1, 0).to(device)
            src_ner_batch = batch['article_ner_ids'].permute(1, 0).to(device)

            batch_size = src_tokens_batch.shape[1]
            for i in range(batch_size):
                # Generate summary for each item in the batch
                generated_text = generate_summary(
                    model,
                    src_tokens_batch[:, i:i+1],
                    src_pos_batch[:, i:i+1],
                    src_dep_batch[:, i:i+1],
                    src_ner_batch[:, i:i+1],
                    sp_tokenizer,
                    MAX_LEN,
                    BOS_IDX,
                    EOS_IDX,
                    device
                )
                generated_summaries.append(generated_text)

                # Decode reference highlight
                ref_ids = trg_tokens_batch[:, i].tolist()
                try:
                    eos_idx = ref_ids.index(EOS_IDX)
                    ref_ids = ref_ids[:eos_idx]
                except ValueError:
                    pass
                if ref_ids and ref_ids[0] == BOS_IDX:
                    ref_ids = ref_ids[1:]
                reference_text = sp_tokenizer.decode_ids(ref_ids)
                reference_summaries.append(reference_text)

    print("\nCalculating metrics...")
    test_metrics = calculate_metrics(reference_summaries, generated_summaries)

    print("\n--- Test Set Evaluation Results ---")

    # Print ROUGE scores
    print("ROUGE Scores:")
    for r_type in ['rouge1', 'rouge2', 'rougeL']:
        print(f"  {r_type.upper()}:")
        print(f"    F1: {test_metrics[r_type + '_f1']:.4f}")
        print(f"    Precision: {test_metrics[r_type + '_precision']:.4f}")
        print(f"    Recall: {test_metrics[r_type + '_recall']:.4f}")

    # Print BERTScore
    print("\nBERTScore:")
    print(f"  Precision: {test_metrics['bertscore_precision']:.4f}")
    print(f"  Recall: {test_metrics['bertscore_recall']:.4f}")
    print(f"  F1: {test_metrics['bertscore_f1']:.4f}")

    # Print METEOR score
    print("\nMETEOR Score: {:.4f}".format(test_metrics['meteor']))

    # Print Readability and Cohesion scores
    print("\nCalculating Readability and Cohesion scores...")
    print("Readability Scores (Average):")
    print(f"  Flesch Reading Ease: {test_metrics['flesch_reading_ease']:.4f}")
    print(f"  Flesch Kincaid Grade: {test_metrics['flesch_kincaid_grade']:.4f}")
    print(f"  Dale Chall Readability Score: {test_metrics['dale_chall_readability_score']:.4f}")
    print(f"  Automated Readability Index: {test_metrics['automated_readability_index']:.4f}")
    print(f"  Coleman Liau Index: {test_metrics['coleman_liau_index']:.4f}")
    print(f"  Linsear Write Formula: {test_metrics['linsear_write_formula']:.4f}")
    print(f"  Gunning Fog Score: {test_metrics['gunning_fog_score']:.4f}")
    print(f"  Smog Index: {test_metrics['smog_index']:.4f}")

    print("\n--- Additional Notes on Cohesion ---")
    print("Cohesion (e.g., measuring semantic similarity between sentences, coreference resolution) is an important aspect of summary quality but is generally not part of standard ROUGE/BLEU/METEOR/BERTScore evaluations.")
    print("Evaluating these often requires more advanced NLP libraries (like spaCy's deeper analysis for discourse structures, or specialized coherence models) or human judgment, and is usually assessed as a separate, more nuanced analysis post-generation.")
    print("This script focuses on standard overlap and semantic similarity metrics, and common readability indices.")


if __name__ == "__main__":
    evaluate_model()


NLTK data downloaded.
Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/test_processed.pt...
Data loaded successfully.
Vocab sizes: Token=30000, POS=50, DEP=45, NER=19
Special IDs: PAD=0, BOS=2, EOS=3


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading model from /kaggle/working/TextSummarization/results/best_model.pt

Starting evaluation on the test set...


Generating summaries:   1%|          | 2/180 [00:15<23:41,  7.99s/it]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import math
import time
import torch.distributions as dist # For sampling in RL

# For evaluation metrics
import evaluate
from rouge_score import rouge_scorer
import nltk
import sentencepiece as spm
# For readability metrics (textstat is already installed via os.system in evaluate.py)
import textstat

# Download NLTK data if not already present for METEOR and BLEU
try:
    nltk.data.find('corpora/wordnet')
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    print("Downloading NLTK data (wordnet, punkt, omw-1.4)... This may take a moment.")
    nltk.download('wordnet')
    nltk.download('punkt')
    nltk.download('omw-1.4')
    print("NLTK data downloaded.")
except Exception as e:
    print(f"An unexpected error occurred during NLTK data check/download: {e}")

# Define constants and paths
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
RESULTS_DIR = f"{BASE_DIR}/results"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model" # To load PAD_ID etc.

# Hyperparameters (you might need to tune these)
EMBEDDING_DIM = 256
ENCODER_HIDDEN_DIM = 512 # Hidden dimension for a single direction of Encoder's LSTM
DECODER_HIDDEN_DIM = ENCODER_HIDDEN_DIM * 2 # Decoder's hidden dim must be double encoder's for direct transfer
ENCODER_LAYERS = 2
DECODER_LAYERS = 4 # Should be equal to ENCODER_LAYERS * 2 for direct transfer of states
ENCODER_DROPOUT = 0.5
DECODER_DROPOUT = 0.5
BATCH_SIZE = 100
N_EPOCHS = 10
LEARNING_RATE = 0.001
CLIP = 1.0 # Gradient clipping
MAX_LEN = 64 # Max sequence length, needs to be consistent with preprocessing

# RL specific hyperparameters
RL_START_EPOCH = 5 # Start RL training after this many MLE epochs
RL_REWARD_METRIC = 'rougeL' # Metric to use for reward (e.g., 'rougeL', 'rouge1', 'rouge2')

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def train_epoch(model: nn.Module,
                data_loader: DataLoader,
                optimizer: optim.Optimizer,
                criterion: nn.Module,
                clip: float,
                pad_idx: int):
    """
    Performs one epoch of standard (MLE) training.

    Args:
        model (nn.Module): The Seq2Seq model.
        data_loader (DataLoader): DataLoader for training data.
        optimizer (optim.Optimizer): Optimizer.
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss).
        clip (float): Gradient clipping value.
        pad_idx (int): Padding token ID.

    Returns:
        float: Average loss for the epoch.
    """
    model.train() # Set model to training mode
    epoch_loss = 0

    for batch in tqdm(data_loader, desc="Training (MLE)"):
        src_tokens = batch['article_ids'].permute(1, 0).to(device) # [seq_len, batch_size]
        trg_tokens = batch['highlight_ids'].permute(1, 0).to(device) # [seq_len, batch_size]
        src_pos = batch['article_pos_ids'].permute(1, 0).to(device)
        src_dep = batch['article_dep_ids'].permute(1, 0).to(device)
        src_ner = batch['article_ner_ids'].permute(1, 0).to(device)

        optimizer.zero_grad() # Clear gradients

        # Output from model: [trg_len, batch_size, vocab_size]
        output = model(src_tokens, src_pos, src_dep, src_ner, trg_tokens)

        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg_tokens = trg_tokens[1:].reshape(-1)

        loss = criterion(output, trg_tokens) # Calculate loss

        loss.backward() # Backpropagation

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip) # Clip gradients

        optimizer.step() # Update model parameters

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)

def sample_summary(model: nn.Module,
                   src_tokens: torch.Tensor,
                   src_pos: torch.Tensor,
                   src_dep: torch.Tensor,
                   src_ner: torch.Tensor,
                   sp_tokenizer: spm.SentencePieceProcessor, # Added tokenizer for generating EOS
                   max_len: int,
                   bos_id: int,
                   eos_id: int,
                   device: torch.device):
    """
    Generates a summary for a single input article by sampling tokens.
    Returns generated text, and log probabilities of sampled tokens.

    Args:
        model (nn.Module): The Seq2Seq model.
        src_tokens (torch.Tensor): Input article token IDs [seq_len, 1] (single sample).
        src_pos (torch.Tensor): Input article POS IDs [seq_len, 1].
        src_dep (torch.Tensor): Input article DEP IDs [seq_len, 1].
        src_ner (torch.Tensor): Input article NER IDs [seq_len, 1].
        sp_tokenizer: SentencePiece tokenizer for decoding.
        max_len (int): Maximum length of the generated summary.
        bos_id (int): Beginning-of-sentence token ID.
        eos_id (int): End-of-sentence token ID.
        device (torch.device): Device to perform computations on.

    Returns:
        tuple: (str: generated summary text, torch.Tensor: log probabilities of sampled tokens)
    """
    model.eval() # Set to eval mode for greedy decoding; for sampling, we still need logits
    with torch.no_grad(): # Initially, run without grad to get encoder_outputs, then enable for decoder
        encoder_outputs, hidden, cell = model.encoder(src_tokens, src_pos, src_dep, src_ner)

        initial_hidden_state_decoder = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1).unsqueeze(0)
        initial_cell_state_decoder = torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1).unsqueeze(0)

        hidden_decoder = initial_hidden_state_decoder.repeat(model.decoder.n_layers, 1, 1)
        cell_decoder = initial_cell_state_decoder.repeat(model.decoder.n_layers, 1, 1)

    # Re-enable grad for the decoder part for RL
    model.train() # Keep model in train mode for RL if we are using policy gradient.
                  # This enables dropout etc. if applicable, and allows gradients.

    input_token = torch.tensor([[bos_id]], device=device) # [1, 1]

    generated_tokens = []
    log_probs_list = []

    for t in range(max_len): # Iterate up to max_len
        output, hidden_decoder, cell_decoder = model.decoder(input_token,
                                                             encoder_outputs,
                                                             hidden_decoder,
                                                             cell_decoder)

        # Get log probabilities of the output distribution
        output_logits = output.squeeze(0) # [vocab_size]
        probabilities = F.softmax(output_logits, dim=-1)
        m = dist.Categorical(probabilities)
        
        # Sample a token
        predicted_token = m.sample()
        log_prob = m.log_prob(predicted_token)

        generated_tokens.append(predicted_token.item())
        log_probs_list.append(log_prob.unsqueeze(0)) # Unsqueeze to keep batch_size dim

        # If EOS is predicted, stop generating
        if predicted_token.item() == eos_id:
            break

        # Use the sampled token as the input for the next step
        input_token = predicted_token.unsqueeze(0).unsqueeze(0) # [1, 1]

    # Decode the generated token IDs
    if generated_tokens and generated_tokens[0] == bos_id:
        generated_tokens = generated_tokens[1:]
    if generated_tokens and generated_tokens[-1] == eos_id:
        generated_tokens = generated_tokens[:-1]
    
    generated_text = sp_tokenizer.decode_ids(generated_tokens)

    # Concatenate log_probs_list, handling empty list
    if log_probs_list:
        log_probs_tensor = torch.cat(log_probs_list).unsqueeze(1) # [seq_len, 1]
    else: # If no tokens were generated (e.g., if max_len was 0 or BOS was EOS)
        log_probs_tensor = torch.tensor([], device=device, dtype=torch.float32).unsqueeze(1)

    return generated_text, log_probs_tensor # Return log_probs as tensor for RL loss calculation


def rl_train_epoch(model: nn.Module,
                   data_loader: DataLoader,
                   optimizer: optim.Optimizer,
                   sp_tokenizer: spm.SentencePieceProcessor,
                   reward_scorer: rouge_scorer.RougeScorer,
                   reward_metric: str,
                   clip: float,
                   max_len: int,
                   bos_id: int,
                   eos_id: int,
                   device: torch.device):
    """
    Performs one epoch of Reinforcement Learning training using REINFORCE.

    Args:
        model (nn.Module): The Seq2Seq model.
        data_loader (DataLoader): DataLoader for training data.
        optimizer (optim.Optimizer): Optimizer.
        sp_tokenizer: SentencePiece tokenizer for decoding.
        reward_scorer: ROUGE scorer instance.
        reward_metric (str): The specific ROUGE metric ('rouge1', 'rouge2', 'rougeL') to use as reward.
        clip (float): Gradient clipping value.
        max_len (int): Maximum length for generated summaries.
        bos_id (int): Beginning-of-sentence token ID.
        eos_id (int): End-of-sentence token ID.
        device (torch.device): Device to perform computations on.

    Returns:
        float: Average RL loss for the epoch.
    """
    model.train() # Set model to training mode
    epoch_rl_loss = 0

    for batch_idx, batch in enumerate(tqdm(data_loader, desc="Training (RL)")):
        src_tokens_batch = batch['article_ids'].permute(1, 0).to(device) # [seq_len, batch_size]
        trg_tokens_batch = batch['highlight_ids'].permute(1, 0).to(device) # [seq_len, batch_size]
        src_pos_batch = batch['article_pos_ids'].permute(1, 0).to(device)
        src_dep_batch = batch['article_dep_ids'].permute(1, 0).to(device)
        src_ner_batch = batch['article_ner_ids'].permute(1, 0).to(device)

        batch_size = src_tokens_batch.shape[1]
        
        # Lists to store log probabilities and rewards for this batch
        batch_log_probs = []
        batch_rewards = []

        optimizer.zero_grad() # Clear gradients for the batch

        # Iterate over each sample in the batch to generate and get rewards
        # (This is inefficient for large batches, but simpler for initial REINFORCE)
        for i in range(batch_size):
            # Generate summary by sampling
            generated_text, log_probs = sample_summary(
                model,
                src_tokens_batch[:, i:i+1],
                src_pos_batch[:, i:i+1],
                src_dep_batch[:, i:i+1],
                src_ner_batch[:, i:i+1],
                sp_tokenizer,
                max_len,
                bos_id,
                eos_id,
                device
            )
            
            # Ensure log_probs is not empty
            if log_probs.numel() == 0:
                reward = torch.tensor(0.0, device=device) # Assign 0 reward for empty summaries
                # Skip if no tokens were generated
                # print(f"Warning: Empty summary generated for sample {i} in batch {batch_idx}. Assigning 0 reward.")
                continue


            # Decode reference highlight for reward calculation
            ref_ids = trg_tokens_batch[:, i].tolist()
            try:
                eos_idx = ref_ids.index(eos_id)
                ref_ids = ref_ids[:eos_idx]
            except ValueError:
                pass
            if ref_ids and ref_ids[0] == bos_id:
                ref_ids = ref_ids[1:]
            reference_text = sp_tokenizer.decode_ids(ref_ids)

            # Calculate ROUGE reward
            score = reward_scorer.score(reference_text, generated_text)
            reward_value = score[reward_metric].fmeasure # Use F-measure of the chosen ROUGE metric
            
            # Store log probabilities and corresponding reward
            # The reward is applied to all log_probs of the generated sequence
            batch_log_probs.append(log_probs.squeeze(1)) # Remove the batch_size=1 dim
            batch_rewards.append(torch.tensor(reward_value, device=device))

        if not batch_log_probs: # If all samples in batch resulted in empty summaries
            # print(f"Warning: No valid summaries generated in batch {batch_idx}. Skipping update.")
            continue


        # Calculate RL loss for the batch
        # We need to sum the log_probs over the sequence for each sample, then apply the reward
        # This is the negative of the policy gradient objective: - sum(log_P(a_t|s_t) * R)
        # where R is the total reward for the episode.
        
        # Ensure log_probs are on CPU for concatenation if they vary in length
        # Or pad them to max length if they are going to be on GPU and in a tensor
        # For simplicity, we process each sample and then stack and concatenate for the loss.
        
        # Flatten the list of log_probs for all samples in the batch
        # Each element in batch_log_probs is a 1D tensor of log_probs for one generated sequence
        combined_log_probs = torch.cat(batch_log_probs)
        
        # Repeat rewards for each token in the corresponding generated sequence
        # This assumes a single reward per sequence (REINFORCE)
        rewards_for_tokens = torch.cat([r.expand_as(lp) for r, lp in zip(batch_rewards, batch_log_probs)])
        
        # Loss is negative log-probability times reward
        rl_loss = - (combined_log_probs * rewards_for_tokens).mean()

        rl_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_rl_loss += rl_loss.item()

    return epoch_rl_loss / len(data_loader) if len(data_loader) > 0 else 0.0


def evaluate_metrics(model: nn.Module,
                     data_loader: DataLoader,
                     sp_tokenizer,
                     bos_id: int,
                     eos_id: int):
    """
    Evaluates the model on the validation/test set by generating summaries
    and calculating summarization metrics.

    Args:
        model (nn.Module): The Seq2Seq model.
        data_loader (DataLoader): DataLoader for evaluation data.
        sp_tokenizer: SentencePiece tokenizer for decoding.
        bos_id (int): Beginning-of-sentence token ID.
        eos_id (int): End-of-sentence token ID.

    Returns:
        dict: Dictionary of calculated summarization metrics.
    """
    model.eval() # Set model to evaluation mode
    generated_summaries = []
    reference_summaries = []

    with torch.no_grad(): # Disable gradient calculations
        for batch in tqdm(data_loader, desc="Generating summaries for evaluation"):
            src_tokens_batch = batch['article_ids'].permute(1, 0).to(device)
            trg_tokens_batch = batch['highlight_ids'].permute(1, 0).to(device)
            src_pos_batch = batch['article_pos_ids'].permute(1, 0).to(device)
            src_dep_batch = batch['article_dep_ids'].permute(1, 0).to(device)
            src_ner_batch = batch['article_ner_ids'].permute(1, 0).to(device)

            batch_size = src_tokens_batch.shape[1]
            for i in range(batch_size):
                # Generate summary for each item in the batch using greedy decoding
                generated_text, _ = sample_summary( # Use sample_summary but discard log_probs
                    model,
                    src_tokens_batch[:, i:i+1],
                    src_pos_batch[:, i:i+1],
                    src_dep_batch[:, i:i+1],
                    src_ner_batch[:, i:i+1],
                    sp_tokenizer,
                    MAX_LEN, # Use MAX_LEN from constants
                    bos_id,
                    eos_id,
                    device
                )
                generated_summaries.append(generated_text)

                # Decode reference highlight
                ref_ids = trg_tokens_batch[:, i].tolist()
                try:
                    eos_idx = ref_ids.index(eos_id)
                    ref_ids = ref_ids[:eos_idx]
                except ValueError:
                    pass
                if ref_ids and ref_ids[0] == bos_id:
                    ref_ids = ref_ids[1:]
                reference_text = sp_tokenizer.decode_ids(ref_ids)
                reference_summaries.append(reference_text)

    print("\nCalculating metrics...")
    all_metrics = calculate_metrics_detailed(reference_summaries, generated_summaries)

    return all_metrics


def calculate_metrics_detailed(references: list[str], predictions: list[str]):
    """
    Calculates ROUGE, BLEU, METEOR, BERTScore, and Readability metrics.
    Uses 'evaluate' library for BLEU, METEOR, BERTScore for consistency.
    """
    metrics = {}

    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_total_scores = {'rouge1': {'fmeasure': 0.0, 'precision': 0.0, 'recall': 0.0},
                          'rouge2': {'fmeasure': 0.0, 'precision': 0.0, 'recall': 0.0},
                          'rougeL': {'fmeasure': 0.0, 'precision': 0.0, 'recall': 0.0}}

    for ref, pred in zip(references, predictions):
        score = scorer.score(ref, pred)
        for k in rouge_total_scores:
            rouge_total_scores[k]['fmeasure'] += score[k].fmeasure
            rouge_total_scores[k]['precision'] += score[k].precision
            rouge_total_scores[k]['recall'] += score[k].recall
    
    for k in rouge_total_scores:
        metrics[k + '_f1'] = rouge_total_scores[k]['fmeasure'] / len(references)
        metrics[k + '_precision'] = rouge_total_scores[k]['precision'] / len(references)
        metrics[k + '_recall'] = rouge_total_scores[k]['recall'] / len(references)


    # BLEU
    bleu_metric = evaluate.load("bleu")
    bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references]) # BLEU expects list of references per prediction
    metrics['bleu'] = bleu_results['bleu']

    # METEOR
    meteor_metric = evaluate.load("meteor")
    meteor_results = meteor_metric.compute(predictions=predictions, references=[[r] for r in references])
    metrics['meteor'] = meteor_results['meteor']

    # BERTScore
    # Using 'distilbert-base-uncased' for faster computation.
    bertscore_metric = evaluate.load("bertscore")
    bertscore_results = bertscore_metric.compute(
        predictions=predictions, references=references, lang="en", model_type="distilbert-base-uncased"
    )
    metrics['bertscore_f1'] = sum(bertscore_results['f1']) / len(bertscore_results['f1'])
    metrics['bertscore_precision'] = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
    metrics['bertscore_recall'] = sum(bertscore_results['recall']) / len(bertscore_results['recall'])


    # Readability Scores (averaged over all generated summaries)
    flesch_reading_ease_scores = []
    flesch_kincaid_grade_scores = []
    dale_chall_readability_scores = []
    automated_readability_index_scores = []
    coleman_liau_index_scores = []
    linsear_write_formula_scores = []
    gunning_fog_scores = []
    smog_index_scores = []

    for pred in predictions:
        if not pred.strip():
            continue

        try:
            flesch_reading_ease_scores.append(textstat.flesch_reading_ease(pred))
            flesch_kincaid_grade_scores.append(textstat.flesch_kincaid_grade(pred))
            dale_chall_readability_scores.append(textstat.dale_chall_readability_score(pred))
            automated_readability_index_scores.append(textstat.automated_readability_index(pred))
            coleman_liau_index_scores.append(textstat.coleman_liau_index(pred))
            linsear_write_formula_scores.append(textstat.linsear_write_formula(pred))
            gunning_fog_scores.append(textstat.gunning_fog(pred))
            smog_index_scores.append(textstat.smog_index(pred))
        except Exception:
            pass # Silently skip if textstat fails for a specific summary

    # Calculate averages, handling empty lists
    metrics['flesch_reading_ease'] = sum(flesch_reading_ease_scores) / len(flesch_reading_ease_scores) if flesch_reading_ease_scores else 0.0
    metrics['flesch_kincaid_grade'] = sum(flesch_kincaid_grade_scores) / len(flesch_kincaid_grade_scores) if flesch_kincaid_grade_scores else 0.0
    metrics['dale_chall_readability_score'] = sum(dale_chall_readability_scores) / len(dale_chall_readability_scores) if dale_chall_readability_scores else 0.0
    metrics['automated_readability_index'] = sum(automated_readability_index_scores) / len(automated_readability_index_scores) if automated_readability_index_scores else 0.0
    metrics['coleman_liau_index'] = sum(coleman_liau_index_scores) / len(coleman_liau_index_scores) if coleman_liau_index_scores else 0.0
    metrics['linsear_write_formula'] = sum(linsear_write_formula_scores) / len(linsear_write_formula_scores) if linsear_write_formula_scores else 0.0
    metrics['gunning_fog_score'] = sum(gunning_fog_scores) / len(gunning_fog_scores) if gunning_fog_scores else 0.0
    metrics['smog_index'] = sum(smog_index_scores) / len(smog_index_scores) if smog_index_scores else 0.0

    return metrics


def main():
    # Load data
    train_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/train_processed.pt")
    val_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/validation_processed.pt")
    test_data = SummarizationDataset(f"{PROCESSED_DATA_DIR}/test_processed.pt")

    # Load SentencePiece tokenizer
    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer.load(TOKENIZER_MODEL_PATH)

    # Get vocabulary sizes and special token IDs from tokenizer
    VOCAB_SIZE = sp_tokenizer.get_piece_size()
    POS_VOCAB_SIZE = train_data.pos_vocab_size
    DEP_VOCAB_SIZE = train_data.dep_vocab_size
    NER_VOCAB_SIZE = train_data.ner_vocab_size
    PAD_IDX = sp_tokenizer.pad_id()
    BOS_IDX = sp_tokenizer.bos_id()
    EOS_IDX = sp_tokenizer.eos_id()


    print(f"Vocab sizes: Token={VOCAB_SIZE}, POS={POS_VOCAB_SIZE}, DEP={DEP_VOCAB_SIZE}, NER={NER_VOCAB_SIZE}")
    print(f"Special IDs: PAD={PAD_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")

    # Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False) # For final evaluation

    # Initialize Encoder, Attention, Decoder, and Seq2Seq Model
    encoder = Encoder(VOCAB_SIZE,
                      POS_VOCAB_SIZE,
                      DEP_VOCAB_SIZE,
                      NER_VOCAB_SIZE,
                      EMBEDDING_DIM,
                      ENCODER_HIDDEN_DIM, # Use ENCODER_HIDDEN_DIM here
                      ENCODER_LAYERS,
                      ENCODER_DROPOUT)

    attention = Attention(ENCODER_HIDDEN_DIM * 2, DECODER_HIDDEN_DIM)

    decoder = Decoder(VOCAB_SIZE,
                      EMBEDDING_DIM,
                      DECODER_HIDDEN_DIM, # Use DECODER_HIDDEN_DIM here
                      DECODER_LAYERS,
                      DECODER_DROPOUT,
                      attention)

    model = Seq2Seq(encoder, decoder, device).to(device)

    # Initialize model weights (optional but good practice)
    def init_weights(m):
        for name, param in m.named_parameters():
            if 'weight' in name:
                nn.init.normal_(param.data, mean=0, std=0.01)
            else:
                nn.init.constant_(param.data, 0)
    model.apply(init_weights)

    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")

    # Optimizer and Loss Function
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    # CrossEntropyLoss ignores specified index when calculating loss, useful for padding
    mle_criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    best_valid_loss = float('inf')
    best_rl_metric_score = -float('inf') # For tracking best RL model

    # Initialize ROUGE scorer for RL reward calculation
    rouge_reward_scorer = rouge_scorer.RougeScorer([RL_REWARD_METRIC], use_stemmer=True)


    print("\nStarting training...")
    for epoch in range(N_EPOCHS):
        start_time = time.time()

        if epoch < RL_START_EPOCH:
            print(f"\n--- Epoch {epoch+1:02} (MLE Training) ---")
            train_loss = train_epoch(model, train_loader, optimizer, mle_criterion, CLIP, PAD_IDX)
            # Evaluate with MLE loss on validation set (metrics are also calculated)
            val_metrics = evaluate_metrics(model, val_loader, sp_tokenizer, BOS_IDX, EOS_IDX)
            valid_loss = val_metrics[RL_REWARD_METRIC + '_f1'] # Use the reward metric's F1 for validation loss tracking (simplified for now)
            
            # For simplicity in output, we won't print full val metrics per MLE epoch
            # just the reward metric for comparison. Full metrics are for test set.
            print(f'Epoch: {epoch+1:02} | Time: {int(time.time() - start_time)}s')
            print(f'\tTrain Loss (MLE): {train_loss:.3f}')
            print(f'\tVal. {RL_REWARD_METRIC.upper()} F1: {val_metrics[RL_REWARD_METRIC + "_f1"]:.4f}')

            # Save model if it improves on validation ROUGE (during MLE pre-training phase)
            if val_metrics[RL_REWARD_METRIC + '_f1'] > best_rl_metric_score: # Use the RL metric to track best model
                best_rl_metric_score = val_metrics[RL_REWARD_METRIC + '_f1']
                torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pt'))
                print(f"\tModel saved for improved {RL_REWARD_METRIC.upper()} F1 score during MLE.")


        else:
            print(f"\n--- Epoch {epoch+1:02} (RL Training) ---")
            rl_loss = rl_train_epoch(model, train_loader, optimizer, sp_tokenizer,
                                     rouge_reward_scorer, RL_REWARD_METRIC, CLIP, MAX_LEN, BOS_IDX, EOS_IDX, device)
            
            # Evaluate with metrics on validation set after RL epoch
            val_metrics = evaluate_metrics(model, val_loader, sp_tokenizer, BOS_IDX, EOS_IDX)
            
            print(f'Epoch: {epoch+1:02} | Time: {int(time.time() - start_time)}s')
            print(f'\tRL Loss: {rl_loss:.3f}')
            print(f'\tVal. {RL_REWARD_METRIC.upper()} F1: {val_metrics[RL_REWARD_METRIC + "_f1"]:.4f}')

            # Save model if it improves on validation ROUGE (during RL training phase)
            if val_metrics[RL_REWARD_METRIC + '_f1'] > best_rl_metric_score:
                best_rl_metric_score = val_metrics[RL_REWARD_METRIC + '_f1']
                torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pt'))
                print(f"\tModel saved for improved {RL_REWARD_METRIC.upper()} F1 score during RL.")

    # Load best model for final evaluation
    model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, 'best_model.pt')))

    print("\n--- Final Evaluation on Test Set ---")
    test_metrics = evaluate_metrics(model, test_loader, sp_tokenizer, BOS_IDX, EOS_IDX)
    
    # Print ROUGE scores
    print("ROUGE Scores:")
    for r_type in ['rouge1', 'rouge2', 'rougeL']:
        print(f"  {r_type.upper()}:")
        print(f"    F1: {test_metrics[r_type + '_f1']:.4f}")
        print(f"    Precision: {test_metrics[r_type + '_precision']:.4f}")
        print(f"    Recall: {test_metrics[r_type + '_recall']:.4f}")

    # Print BERTScore
    print("\nBERTScore:")
    print(f"  Precision: {test_metrics['bertscore_precision']:.4f}")
    print(f"  Recall: {test_metrics['bertscore_recall']:.4f}")
    print(f"  F1: {test_metrics['bertscore_f1']:.4f}")

    # Print METEOR score
    print("\nMETEOR Score: {:.4f}".format(test_metrics['meteor']))

    # Print Readability and Cohesion scores
    print("\nReadability Scores (Average):")
    print(f"  Flesch Reading Ease: {test_metrics['flesch_reading_ease']:.4f}")
    print(f"  Flesch Kincaid Grade: {test_metrics['flesch_kincaid_grade']:.4f}")
    print(f"  Dale Chall Readability Score: {test_metrics['dale_chall_readability_score']:.4f}")
    print(f"  Automated Readability Index: {test_metrics['automated_readability_index']:.4f}")
    print(f"  Coleman Liau Index: {test_metrics['coleman_liau_index']:.4f}")
    print(f"  Linsear Write Formula: {test_metrics['linsear_write_formula']:.4f}")
    print(f"  Gunning Fog Score: {test_metrics['gunning_fog_score']:.4f}")
    print(f"  Smog Index: {test_metrics['smog_index']:.4f}")

    print("\n--- Additional Notes on Cohesion ---")
    print("Cohesion (e.g., measuring semantic similarity between sentences, coreference resolution) is an important aspect of summary quality but is generally not part of standard ROUGE/BLEU/METEOR/BERTScore evaluations.")
    print("Evaluating these often requires more advanced NLP libraries (like spaCy's deeper analysis for discourse structures, or specialized coherence models) or human judgment, and is usually assessed as a separate, more nuanced analysis post-generation.")
    print("This script focuses on standard overlap and semantic similarity metrics, and common readability indices.")


if __name__ == "__main__":
    main()



NLTK data downloaded.
Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/train_processed.pt...
Data loaded successfully.
Loading data from /kaggle/working/TextSummarization/data/processed/validation_processed.pt...
Data loaded successfully.
Loading data from /kaggle/working/TextSummarization/data/processed/test_processed.pt...
Data loaded successfully.
Vocab sizes: Token=30000, POS=50, DEP=45, NER=19
Special IDs: PAD=0, BOS=2, EOS=3


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Model has 131,752,714 trainable parameters

Starting training...

--- Epoch 01 (MLE Training) ---


Generating summaries for evaluation:  87%|████████▋ | 117/134 [23:35<03:17, 11.61s/it]